In [ ]:
# CELL 1 — configuration (edit this before running)
from pathlib import Path
from dataclasses import dataclass

@dataclass(frozen=True)
class UserConfig:
    RAW_CSV_PATH: str = '/kaggle/input/competitions/nexar-collision-prediction/train.csv'
    RAW_VIDEO_DIR: str = '/kaggle/input/competitions/nexar-collision-prediction/train'
    WORK_DIR: str = '/kaggle/working/nexar_multiseed'
    # One-click is the default. Staged mode remains available for advanced recovery.
    EXECUTION_MODE: str = 'one_click'  # 'one_click' or 'staged'
    STAGES: tuple = ()  # staged mode only; leave empty for one-click
    RUN_BENCHMARKS: bool = True  # real measured Table III + latency figure
    RUN_CACHE_PROFILE: bool = False  # extra throughput diagnostic
    DRY_RUN: bool = False           # True = small pilot, never submit pilot results as confirmatory
    MAX_PILOT_VIDEOS: int = 72
    SEED: int = 42  # fixed DATA/SPLIT seed; do not change across model seeds
    TRAIN_SEEDS: tuple = (42, 123, 2026)  # predeclared seeds; never pick the best TEST seed
    RUN_SEEDS: tuple = (42, 123, 2026)  # one-click requires ALL declared seeds
    CACHE_BACKEND: str = 'uint8_npy'  # 4 x 16 x 224 x 224 x RGB uint8, 1 .npy/video
    CACHE_RESERVE_GIB: float = 1.5  # keep free space for checkpoints/logs
    CACHE_MAX_GIB: float = 16.0
    PERSISTENT_WORKERS: bool = True
    PREFETCH_FACTOR: int = 2
    PIN_MEMORY: bool = True
    GPU_AUGMENT: bool = True
    CUDA_TF32: bool = True
    STRICT_DETERMINISM: bool = False
    TRAIN_FRAC: float = 0.60
    TUNE_FRAC: float = 0.20
    TEST_FRAC: float = 0.20
    HORIZONS: tuple = (0.5, 1.0, 1.5)
    CLIP_SECONDS: float = 3.0
    FRAMES: int = 16
    IMAGE_SIZE: int = 224
    VIDEO_WORKERS: int = 3
    REQUIRE_FRAME_PTS: bool = True  # refuse unverified frame timestamps; vital for strictly pre-event claims
    NUM_WORKERS: int = 4
    TRAIN_BATCH_SIZE: int = 2  # 16 x 224x224 frame models can be memory-hungry
    EVAL_BATCH_SIZE: int = 4
    EPOCHS: int = 40
    EVAL_EVERY: int = 2
    LR: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    ACCUM_STEPS: int = 4
    AMP_ON_CUDA: bool = True
    PRETRAINED: bool = True
    # A failed pretrained-weight load is a visible failure; set PRETRAINED=False to opt out.
    REGIMES: tuple = ('EC', 'PE')
    ENCODERS: tuple = ('EfficientNet-B0+BiGRU', 'MobileNetV3+TSM', 'Compact3D-X3D-inspired')
    FUSION_GRID_STEP: float = 0.05
    FPR_TUNING_TARGET: float = 0.10
    BOOTSTRAPS: int = 2000
    PERMUTATIONS: int = 1999
    LATENCY_REPEATS: int = 30
    FINAL_TEST_UNLOCK: bool = False  # staged mode only; one-click gate opens after verified locks
    ALLOW_OVERWRITE_FINAL: bool = False

CFG = UserConfig()
ROOT = Path(CFG.WORK_DIR)
for part in ('manifests','uint8_cache','tables','figures','stats','runs'):
    (ROOT / part).mkdir(parents=True, exist_ok=True)
assert abs(CFG.TRAIN_FRAC + CFG.TUNE_FRAC + CFG.TEST_FRAC - 1.) < 1e-9
assert CFG.HORIZONS == tuple(sorted(CFG.HORIZONS))
print('Mode:', CFG.EXECUTION_MODE, 'stages:', CFG.STAGES, 'root:', ROOT, 'dry run:', CFG.DRY_RUN)
if CFG.DRY_RUN: print('PILOT ONLY: results must not be described as held-out confirmation.')
assert len(set(CFG.TRAIN_SEEDS)) == len(CFG.TRAIN_SEEDS) >= 2, 'Prespecify at least two unique training seeds'
assert set(CFG.RUN_SEEDS).issubset(set(CFG.TRAIN_SEEDS)) and len(CFG.RUN_SEEDS)>0
assert CFG.CACHE_BACKEND == 'uint8_npy', 'Use uint8_npy; old MP4 cache is not equivalent to the new pipeline'

assert CFG.EXECUTION_MODE in ('one_click','staged')
if CFG.EXECUTION_MODE == 'one_click':
    assert CFG.STAGES == (), 'One-click mode owns all stages; leave STAGES empty.'
    assert CFG.RUN_SEEDS == CFG.TRAIN_SEEDS, 'Do not omit or reorder prespecified model seeds.'
    assert not CFG.FINAL_TEST_UNLOCK, 'One-click test access is granted only by verified frozen locks.'
    assert not CFG.ALLOW_OVERWRITE_FINAL, 'Never overwrite completed confirmatory test results.'


In [ ]:
# CELL 2 — imports, deterministic utilities, environment
import json, time, copy, random, hashlib, platform, gc, shutil, multiprocessing
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.special import expit
from scipy.stats import binomtest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (average_precision_score, roc_auc_score, confusion_matrix,
                             f1_score, matthews_corrcoef, balanced_accuracy_score,
                             brier_score_loss, log_loss)
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
try:
    import torchvision
    from torchvision import models
except Exception as e:
    raise RuntimeError('torchvision is required; fix your Kaggle PyTorch/torchvision versions first') from e

# Dedicated Kaggle GPU/CPU pipeline (TPU/XLA is not part of this experiment).
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = CFG.NUM_WORKERS
cv2.setNumThreads(0)

def seed_all(seed):
    random.seed(int(seed)); np.random.seed(int(seed)); torch.manual_seed(int(seed))
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))
seed_all(CFG.SEED)

def save_json(obj, path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=lambda x: float(x) if isinstance(x,np.floating) else (int(x) if isinstance(x,np.integer) else str(x))))
    tmp.replace(path)

def sha256_bytes(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def clip_key(h): return 'EC' if h == 'EC' else f'H{float(h):.1f}'
def clean_model_name(s): return ''.join(c if c.isalnum() else '_' for c in s).strip('_')
def model_key(regime, name): return f'{regime}__{clean_model_name(name)}'

def normalize_id(value):
    s=str(value).strip()
    if s.endswith('.0') and s[:-2].isdigit(): s=s[:-2]
    return s.zfill(5) if s.isdigit() else s

print({'device': str(DEVICE),'torch': torch.__version__, 'torchvision': torchvision.__version__, 'python': platform.python_version()})
# Performance settings: float32 matmul TF32 may slightly affect reproducibility; explicitly log it.
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = CFG.CUDA_TF32 and not CFG.STRICT_DETERMINISM
    torch.backends.cudnn.allow_tf32 = CFG.CUDA_TF32 and not CFG.STRICT_DETERMINISM
    torch.backends.cudnn.benchmark = not CFG.STRICT_DETERMINISM
if CFG.STRICT_DETERMINISM:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

def worker_seed_init(worker_id):
    # Worker-level numpy and Python RNG when any external operation uses them.
    seed = int(torch.initial_seed()) % (2**32)
    np.random.seed(seed); random.seed(seed)

def activate_seed(seed):
    global RUN_SEED, RUN_ROOT
    if seed not in CFG.TRAIN_SEEDS: raise ValueError(f'Unregistered seed {seed}')
    RUN_SEED=int(seed)
    RUN_ROOT=ROOT/'runs'/f'seed_{RUN_SEED}'
    for p in ('checkpoints','predictions','selection','tables','figures','stats','logs'):
        (RUN_ROOT/p).mkdir(parents=True,exist_ok=True)
    seed_all(RUN_SEED)
    print('Active independent training seed:',RUN_SEED,'model outputs:',RUN_ROOT)

activate_seed(CFG.RUN_SEEDS[0])


In [ ]:
# CELL 3 — audit raw video availability and create one locked source-video split
WINDOWS = ('EC',) + tuple(CFG.HORIZONS)

def intervals(anchor):
    anchor=float(anchor)
    return {'EC': (anchor-CFG.CLIP_SECONDS/2, anchor+CFG.CLIP_SECONDS/2),
            **{h: (anchor-h-CFG.CLIP_SECONDS, anchor-h) for h in CFG.HORIZONS}}

def valid_anchor(anchor,duration):
    return all(start >= -1e-6 and end <= duration + 1e-6 and end>start
               for start,end in intervals(anchor).values())

def probe_video(video_path):
    cap=cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return (np.nan,np.nan,False)
    fps=float(cap.get(cv2.CAP_PROP_FPS))
    frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    good=np.isfinite(fps) and fps>0 and frames>0
    return (fps, frames/fps if good else np.nan, bool(good))

def build_audit():
    csv_path=Path(CFG.RAW_CSV_PATH)
    if not csv_path.exists(): raise FileNotFoundError('Attach raw Nexar competition dataset and update RAW_CSV_PATH: '+str(csv_path))
    raw=pd.read_csv(csv_path,dtype={'id':str})
    required={'id','target','time_of_event'}
    if not required.issubset(raw.columns): raise ValueError(f'Missing columns {required-set(raw.columns)}; actual columns: {list(raw.columns)}')
    raw['video_id']=raw['id'].map(normalize_id)
    raw['target']=pd.to_numeric(raw['target'],errors='raise').astype(int)
    if not set(raw.target.unique()).issubset({0,1}): raise ValueError('Unexpected targets; check Nexar label conventions')
    if raw.video_id.duplicated().any(): raise ValueError('Duplicate video IDs in raw metadata; audit before splitting')
    raw['event_time']=pd.to_numeric(raw['time_of_event'],errors='coerce')
    raw['alert_time']=pd.to_numeric(raw.get('time_of_alert',pd.Series(index=raw.index,dtype=float)),errors='coerce')
    if CFG.DRY_RUN:
        pos=raw[raw.target==1].sample(n=min(CFG.MAX_PILOT_VIDEOS//2,int(sum(raw.target==1))),random_state=CFG.SEED)
        neg=raw[raw.target==0].sample(n=min(CFG.MAX_PILOT_VIDEOS//2,int(sum(raw.target==0))),random_state=CFG.SEED)
        raw=pd.concat([pos,neg]).sort_values('video_id').reset_index(drop=True)
    paths=[Path(CFG.RAW_VIDEO_DIR)/f'{vid}.mp4' for vid in raw.video_id]
    raw['video_path']=list(map(str,paths))
    fps=[]; dur=[]; readable=[]
    with ThreadPoolExecutor(max_workers=CFG.VIDEO_WORKERS) as ex:
        results=list(tqdm(ex.map(probe_video,paths),total=len(paths),desc='Video metadata audit'))
    for f,d,o in results: fps.append(f); dur.append(d); readable.append(o)
    raw['fps']=fps; raw['duration']=dur; raw['readable']=readable
    raw['eligible_positive']=raw.apply(lambda r: valid_anchor(r.event_time,r.duration) if r.target==1 and r.readable and np.isfinite(r.event_time) else False,axis=1)
    raw['eligible_negative']=raw.apply(lambda r: bool(r.target==0 and r.readable and r.duration>=CFG.CLIP_SECONDS+max(CFG.HORIZONS)+CFG.CLIP_SECONDS/2),axis=1)
    raw['eligible']=raw.eligible_positive|raw.eligible_negative
    raw.to_csv(ROOT/'manifests'/'dataset_audit.csv',index=False)
    eligible=raw[raw.eligible].copy()
    if eligible.target.nunique()!=2 or len(eligible)<30: raise RuntimeError('Too few eligible positive/negative videos; inspect dataset_audit.csv')
    # Split stratification is guaranteed on the binary label; subtype can be reported as a secondary attribute.
    a,b=train_test_split(eligible,test_size=1-CFG.TRAIN_FRAC,stratify=eligible.target,random_state=CFG.SEED)
    tune_frac_of_remainder=CFG.TUNE_FRAC/(CFG.TUNE_FRAC+CFG.TEST_FRAC)
    tune,test=train_test_split(b,train_size=tune_frac_of_remainder,stratify=b.target,random_state=CFG.SEED+1)
    eligible.loc[a.index,'split']='train'; eligible.loc[tune.index,'split']='tune'; eligible.loc[test.index,'split']='test'
    assert eligible.groupby('video_id').split.nunique().max()==1
    # Negative pseudo-anchors sampled from TRAIN positive event-time distribution (not from held-out labels).
    train_evt=eligible[(eligible.split=='train')&(eligible.target==1)].event_time.dropna().to_numpy(float)
    rng=np.random.default_rng(CFG.SEED)
    anchors=[]; anchor_source=[]
    for _,r in eligible.iterrows():
        if r.target==1:
            anchors.append(float(r.event_time)); anchor_source.append('event_timestamp')
        else:
            choices=train_evt[[valid_anchor(t,float(r.duration)) for t in train_evt]]
            if not len(choices): raise RuntimeError('No valid train-positive pseudo-anchor for normal video '+r.video_id)
            anchors.append(float(rng.choice(choices))); anchor_source.append('train_positive_anchor_distribution')
    eligible['anchor_time']=anchors; eligible['anchor_source']=anchor_source
    eligible['protocol_version']='NEXAR_TVDR_V1'
    eligible=eligible.sort_values(['split','video_id']).reset_index(drop=True)
    assert eligible.video_id.is_unique and eligible.apply(lambda r:valid_anchor(r.anchor_time,r.duration),axis=1).all()
    eligible.to_csv(ROOT/'manifests'/'split_manifest.csv',index=False)
    stats=eligible.groupby(['split','target']).size().unstack(fill_value=0).to_dict()
    save_json({'protocol':'NEXAR_TVDR_V1','seed':CFG.SEED,'split_counts':stats,
               'eligible':int(len(eligible)),'excluded':int(len(raw)-len(eligible)),
               'total_raw':int(len(raw)),'source_video_disjoint':True,
               'note':'Source-video IDs disjoint; near-duplicate scene matching across IDs not proven.'},ROOT/'manifests'/'audit_summary.json')
    print('Eligible cohort:',len(eligible),'exclusions:',len(raw)-len(eligible))
    print(eligible.groupby(['split','target']).size()); return eligible

manifest_file=ROOT/'manifests'/'split_manifest.csv'
if 'audit' in CFG.STAGES:
    if manifest_file.exists():
        raise RuntimeError('Split manifest already exists. Refusing to silently rewrite a frozen split. Use the existing manifest or intentionally start a NEW WORK_DIR.')
    MANIFEST=build_audit()
elif manifest_file.exists(): MANIFEST=pd.read_csv(manifest_file,dtype={'video_id':str})
else: MANIFEST=None

In [ ]:
# CELL 4 — manifest unit checks (fast: no GPU, no raw videos needed)
def protocol_self_checks():
    w=intervals(10.0)
    assert w['EC']==(8.5,11.5)
    assert w[0.5]==(6.5,9.5) and w[1.5]==(5.5,8.5)
    for h in CFG.HORIZONS: assert w[h][1]<=10-h
    assert valid_anchor(6.0,11.0)
    assert not valid_anchor(2.0,11.0)
    if MANIFEST is not None:
        assert MANIFEST.video_id.is_unique
        assert MANIFEST.groupby('video_id').split.nunique().max()==1
        assert MANIFEST.target.isin([0,1]).all()
        assert MANIFEST.apply(lambda r:valid_anchor(r.anchor_time,r.duration),axis=1).all()
    print('Protocol and source-ID checks PASSED.')
protocol_self_checks()

In [ ]:
# CELL 5 — extract lossless uint8 RGB cache, ONE .npy/video for all 4 windows
# Cache is shared among ALL training seeds. Memory mapping prevents 13+ GiB RAM loads.
# The old MPEG-4 clip cache was lossy and repeatedly decoded at every epoch.
CACHE_DIR=ROOT/'uint8_cache'
CACHE_DIR.mkdir(parents=True,exist_ok=True)
WINDOW_ORDER=list(WINDOWS)
WINDOW_TO_INDEX={clip_key(h):i for i,h in enumerate(WINDOW_ORDER)}
CACHE_SHAPE=(len(WINDOW_ORDER),CFG.FRAMES,CFG.IMAGE_SIZE,CFG.IMAGE_SIZE,3)
BYTES_PER_VIDEO=int(np.prod(CACHE_SHAPE))  # uint8 -> 1 byte/pixel/channel

def uint8_cache_path(video_id):
    return CACHE_DIR/f'{normalize_id(video_id)}.npy'

CACHE_SCHEMA = 'NEXAR_UINT8_CFR_PTS_V2'

def cache_metadata(row):
    return {
        'cache_schema': CACHE_SCHEMA,
        'video_id': str(row.video_id),
        'video_path': str(row.video_path),
        'anchor_time': float(row.anchor_time),
        'fps': float(row.fps),
        'duration': float(row.duration),
        'frames': CFG.FRAMES,
        'image_size': CFG.IMAGE_SIZE,
        'clip_seconds': CFG.CLIP_SECONDS,
        'horizons': list(CFG.HORIZONS),
        'windows': [clip_key(h) for h in WINDOW_ORDER],
        'timestamp_verification': bool(CFG.REQUIRE_FRAME_PTS),
        'sampling': 'floor(center_seconds * nominal_fps); decoded_frame_PTS_checked',
    }

def cache_meta_path(video_id):
    return uint8_cache_path(video_id).with_suffix('.meta.json')

def validate_uint8_file(path):
    try:
        arr = np.load(path, mmap_mode='r', allow_pickle=False)
        try:
            return arr.dtype == np.uint8 and arr.shape == CACHE_SHAPE
        finally:
            del arr
    except (OSError, ValueError, EOFError):
        return False

def valid_cached_video(row):
    target = uint8_cache_path(row.video_id)
    sidecar = cache_meta_path(row.video_id)
    if not target.is_file() or not sidecar.is_file():
        return False
    try:
        recorded = json.loads(sidecar.read_text())
        if recorded.get('protocol') != cache_metadata(row):
            return False
        # If source videos are mounted, detect in-place replacements via stat fingerprint.
        source = Path(row.video_path)
        if source.is_file():
            st = source.stat()
            if recorded.get('source_stat') != {'size': st.st_size, 'mtime_ns': st.st_mtime_ns}:
                return False
        return validate_uint8_file(target)
    except (OSError, ValueError, KeyError, TypeError, json.JSONDecodeError):
        return False

def cache_disk_preflight(manifest):
    unique_videos=manifest.video_id.nunique()
    expected=unique_videos*BYTES_PER_VIDEO
    actual=sum(p.stat().st_size for p in CACHE_DIR.glob('*.npy'))
    missing=sum(not valid_cached_video(r) for r in manifest.itertuples(index=False))
    add_bytes=missing*(BYTES_PER_VIDEO+256)
    free=shutil.disk_usage(CACHE_DIR).free
    gib=1024**3
    print(f'uint8 cache expected: {expected/gib:.2f} GiB; existing: {actual/gib:.2f} GiB; '
          f'additional required: {add_bytes/gib:.2f} GiB; free disk: {free/gib:.2f} GiB')
    if expected/gib > CFG.CACHE_MAX_GIB:
        raise RuntimeError('uint8 cache estimated above CACHE_MAX_GIB; reduce image resolution or plan disk space intentionally')
    if add_bytes+CFG.CACHE_RESERVE_GIB*gib>free:
        raise RuntimeError('Insufficient cache disk space. uint8 cache is intentionally lossless; do not silently fall back to compressed MP4. '
                           'Attach an external persistent cache or free Kaggle working space.')
    return expected

def extract_frames_from_video(path, frames_by_window, image_size, time_windows, nominal_fps):
    reverse = {}
    for key, indices in frames_by_window.items():
        for j, idx in enumerate(indices):
            reverse.setdefault(int(idx), []).append((key, j))
    requested = sorted(reverse)
    if not requested:
        raise RuntimeError('No requested frames')
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise IOError(f'Cannot open {path}')
    found = {}
    try:
        cap.set(cv2.CAP_PROP_POS_FRAMES, requested[0])
        pos = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        if pos > requested[0]:
            raise IOError(f'Decoder skipped first requested frame: {path}')
        while pos <= requested[-1]:
            ok, frame = cap.read()
            if not ok:
                raise IOError(f'Decode ended at frame {pos}: {path}')
            if pos in reverse:
                pts = float(cap.get(cv2.CAP_PROP_POS_MSEC)) / 1000.0
                if CFG.REQUIRE_FRAME_PTS and (not np.isfinite(pts) or (pos > 0 and pts <= 0)):
                    raise RuntimeError(f'Unverifiable frame presentation time at index {pos}: {path}. '
                                       'Use a decoder with actual PTS; do not claim strict pre-event timing.')
                if CFG.REQUIRE_FRAME_PTS:
                    # Timestamps can differ slightly from nominal frame centers; however
                    # no selected frame is allowed to reach or exceed its window cutoff.
                    for key, _ in reverse[pos]:
                        start, end = time_windows[key]
                        tolerance = max(2.0 / nominal_fps, 0.06)
                        if pts < start - tolerance or pts >= end - 1e-6:
                            raise RuntimeError(f'Sampled frame PTS {pts:.6f} outside '
                                               f'window [{start:.6f}, {end:.6f}) for {key}: {path}')
                img = cv2.cvtColor(cv2.resize(frame, (image_size, image_size),
                                                interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
                for key, j in reverse[pos]:
                    found[(key, j)] = img
            pos += 1
    finally:
        cap.release()
    need = sum(len(v) for v in frames_by_window.values())
    if len(found) != need:
        raise IOError(f'Incomplete cached frames: expected={need}, got={len(found)} at {path}')
    return np.stack([np.stack([found[(key, j)] for j in range(len(frames_by_window[key]))])
                     for key in WINDOW_TO_INDEX], axis=0).astype(np.uint8, copy=False)

def extract_one_video(row):
    vid = str(row.video_id)
    target = uint8_cache_path(vid)
    anchors = intervals(row.anchor_time)
    if not valid_cached_video(row):
        indices = {}
        for h, (start, end) in anchors.items():
            centers = start + (np.arange(CFG.FRAMES) + .5) * (end-start) / CFG.FRAMES
            idx = np.floor(centers * float(row.fps)).astype(int)
            if not np.all((centers < end) & (idx >= 0)):
                raise RuntimeError(f'Invalid temporal sampling for {vid}/{h}')
            indices[clip_key(h)] = idx
        time_windows = {clip_key(h): bounds for h, bounds in anchors.items()}
        stack = extract_frames_from_video(row.video_path, indices, CFG.IMAGE_SIZE,
                                          time_windows, float(row.fps))
        if stack.shape != CACHE_SHAPE:
            raise AssertionError(f'Bad cache shape for {vid}: {stack.shape}')
        tmp = target.with_suffix('.part.npy')
        try:
            np.save(tmp, stack, allow_pickle=False)
            if not validate_uint8_file(tmp):
                raise RuntimeError(f'Cache verification failed: {tmp}')
            tmp.replace(target)
            st = Path(row.video_path).stat()
            save_json({'protocol': cache_metadata(row),
                       'source_stat': {'size': st.st_size, 'mtime_ns': st.st_mtime_ns}},
                      cache_meta_path(vid))
        finally:
            tmp.unlink(missing_ok=True)
    return [(vid, row.split, int(row.target), clip_key(h), WINDOW_TO_INDEX[clip_key(h)],
             float(start), float(end), str(target),
             float(row.event_time) if row.target else np.nan,
             float(row.alert_time) if row.target and pd.notna(row.alert_time) else np.nan)
            for h, (start, end) in anchors.items()]

def build_clips():
    if MANIFEST is None: raise RuntimeError('Run audit first')
    cache_disk_preflight(MANIFEST)
    rows=[];failures=[]
    with ThreadPoolExecutor(max_workers=CFG.VIDEO_WORKERS) as ex:
        futures={ex.submit(extract_one_video,r):str(r.video_id) for r in MANIFEST.itertuples(index=False)}
        for fut in tqdm(as_completed(futures),total=len(futures),desc='Cache uint8 video windows'):
            try: rows.extend(fut.result())
            except Exception as err: failures.append({'video_id':futures[fut],'error':str(err)})
    if failures:
        pd.DataFrame(failures).to_csv(ROOT/'manifests'/'extraction_failures.csv',index=False)
        raise RuntimeError(f'{len(failures)} extraction failures; see extraction_failures.csv')
    cols=['video_id','split','target','horizon','window_index','clip_start','clip_end','path','event_time','alert_time']
    df=pd.DataFrame(rows,columns=cols).sort_values(['split','video_id','window_index'])
    df['cache_dtype']='uint8';df['cache_backend']='npy_memmap_rgb'
    if not df.groupby('video_id').horizon.nunique().eq(len(WINDOWS)).all(): raise AssertionError('Missing horizon')
    if not df.groupby('video_id').split.nunique().eq(1).all(): raise AssertionError('Video crosses splits')
    df.to_csv(ROOT/'manifests'/'clip_manifest.csv',index=False)
    save_json({'backend':'uint8_npy','cache_schema':CACHE_SCHEMA,'shape':CACHE_SHAPE,'dtype':'uint8','num_videos':int(df.video_id.nunique()),
               'estimated_cache_gib':df.video_id.nunique()*BYTES_PER_VIDEO/(1024**3),
               'rgb':True,'compression':'none','split_seed':CFG.SEED,'training_seeds':CFG.TRAIN_SEEDS},
              ROOT/'manifests'/'cache_spec.json')
    print('Cached',df.video_id.nunique(),'videos,',len(df),'windows; all shared across training seeds')
    return df

clip_manifest_file=ROOT/'manifests'/'clip_manifest.csv'
if 'extract' in CFG.STAGES:
    if clip_manifest_file.exists():
        raise RuntimeError('Existing clip manifest; cache protocol cannot silently change. Start a fresh WORK_DIR or use the existing cache.')
    CLIPS=build_clips()
elif clip_manifest_file.exists():
    CLIPS=pd.read_csv(clip_manifest_file,dtype={'video_id':str})
    if 'window_index' not in CLIPS or not set(CLIPS.cache_dtype)=={'uint8'}:
        raise RuntimeError('Legacy lossy clip manifest detected. Start a fresh WORK_DIR for uint8 cache.')
    spec_path = ROOT/'manifests'/'cache_spec.json'
    if not spec_path.exists() or json.loads(spec_path.read_text()).get('cache_schema') != CACHE_SCHEMA:
        raise RuntimeError('Old cache protocol detected: start a new WORK_DIR and re-extract frames.')
    if MANIFEST is None:
        raise RuntimeError('Missing split manifest; cannot validate cache provenance')
    records = {str(r.video_id): r for r in MANIFEST.itertuples(index=False)}
    invalid = [vid for vid in CLIPS.video_id.unique() if vid not in records or not valid_cached_video(records[vid])]
    if invalid:
        raise RuntimeError(f'{len(invalid)} stale or unverifiable cached videos (first: {invalid[:3]}); '
                           'start with a clean WORK_DIR or repair extraction before continuing.')
else: CLIPS=None


In [ ]:
# CELL 6 — memory-mapped uint8 dataset, worker-safe epoch rotation, pinned loader
# Source frames remain uint8 until H2D transfer. Normalize and augment on CUDA, not workers.
NORM_MEAN=torch.tensor([.485,.456,.406],dtype=torch.float32).view(1,1,3,1,1)
NORM_STD=torch.tensor([.229,.224,.225],dtype=torch.float32).view(1,1,3,1,1)

NORM_MEAN_DEVICE=NORM_MEAN.to(DEVICE)
NORM_STD_DEVICE=NORM_STD.to(DEVICE)

def load_uint8_window(path, index):
    arr = np.load(path, mmap_mode='r', allow_pickle=False)
    if arr.shape != CACHE_SHAPE or arr.dtype != np.uint8:
        raise IOError(f'Invalid uint8 cache: {path}')
    clip = np.array(arr[int(index)].transpose(0, 3, 1, 2), order='C', copy=True)
    return torch.from_numpy(clip)

class StudyDataset(Dataset):
    def __init__(self,clip_manifest,split,regime,horizon=None,training=False,seed=None):
        self.regime=regime;self.training=training;self.seed=int(RUN_SEED if seed is None else seed)
        self._epoch=multiprocessing.Value('i',0)
        eligible=clip_manifest[clip_manifest.split==split].copy()
        if regime=='EC': eligible=eligible[eligible.horizon=='EC']
        elif training and regime=='PE': eligible=eligible[eligible.horizon==clip_key(CFG.HORIZONS[0])]
        elif regime=='PE':
            if horizon not in CFG.HORIZONS: raise ValueError('PE evaluation requires a horizon')
            eligible=eligible[eligible.horizon==clip_key(horizon)]
        else: raise ValueError(regime)
        self.rows=eligible.sort_values('video_id').reset_index(drop=True)
        if self.rows.empty or not self.rows.video_id.is_unique: raise ValueError('Missing/duplicate videos')
        self.examples=list(self.rows[['video_id','path','window_index','target']].itertuples(index=False,name=None))
        self.lookup={(str(r.video_id),str(r.horizon)):int(r.window_index)
                     for r in clip_manifest.itertuples(index=False)}
    def set_epoch(self,epoch):
        with self._epoch.get_lock(): self._epoch.value=int(epoch)
    def __len__(self): return len(self.rows)
    def __getitem__(self,idx):
        vid, path, window, target = self.examples[idx]
        if self.training and self.regime == 'PE':
            epoch = self._epoch.value
            horizon = CFG.HORIZONS[(idx + epoch + self.seed) % len(CFG.HORIZONS)]
            window = self.lookup[(str(vid), clip_key(horizon))]
        return load_uint8_window(path, window), int(target)

def get_loader(split,regime,horizon=None,training=False,seed=None):
    if CLIPS is None: raise RuntimeError('Missing uint8 cache manifest')
    ds=StudyDataset(CLIPS,split,regime,horizon,training=training,seed=seed)
    workers=NUM_WORKERS
    opts=dict(dataset=ds,batch_size=CFG.TRAIN_BATCH_SIZE if training else CFG.EVAL_BATCH_SIZE,
              shuffle=training,num_workers=workers,
              pin_memory=(DEVICE.type=='cuda' and CFG.PIN_MEMORY),
              persistent_workers=bool(CFG.PERSISTENT_WORKERS and workers>0),drop_last=False,
              worker_init_fn=worker_seed_init if workers else None)
    if workers:opts['prefetch_factor']=CFG.PREFETCH_FACTOR
    if training:
        gen=torch.Generator();gen.manual_seed(ds.seed)
        opts['generator']=gen  # worker RNG, separate from shuffle
        opts['shuffle']=False
        opts['sampler']=RandomSampler(ds, generator=torch.Generator())
    return DataLoader(**opts)

def preprocess_uint8_batch(batch,training=False):
    if batch.dtype != torch.uint8: raise TypeError('Expected pinned uint8 clip tensor')
    x=batch.to(DEVICE,non_blocking=(DEVICE.type=='cuda'))
    # Conversion/normalization on GPU saves ~4x H2D bytes versus CPU float32 decoding.
    x=x.float().div_(255.)
    if training and CFG.GPU_AUGMENT:
        b=x.shape[0]
        # All frames from one video receive identical flip/brightness/contrast parameters.
        flip=torch.rand(b,device=DEVICE)<.5
        x=torch.where(flip.view(-1,1,1,1,1),x.flip(-1),x)
        change=torch.rand(b,device=DEVICE)<.5
        contrast=torch.empty(b,device=DEVICE).uniform_(.85,1.15)
        brightness=torch.empty(b,device=DEVICE).uniform_(-15/255,15/255)
        contrast=torch.where(change,contrast,torch.ones_like(contrast))
        brightness=torch.where(change,brightness,torch.zeros_like(brightness))
        x.mul_(contrast.view(-1,1,1,1,1)).add_(brightness.view(-1,1,1,1,1)).clamp_(0,1)
        noise=torch.rand((b,1,1,1,1),device=DEVICE)<.15
        sigma=torch.empty((b,1,1,1,1),device=DEVICE).uniform_(2/255,7/255)
        x=(x+noise*torch.randn_like(x)*sigma).clamp_(0,1)
    return (x-NORM_MEAN_DEVICE)/NORM_STD_DEVICE

if CLIPS is not None:
    assert CLIPS.groupby('video_id').split.nunique().max()==1
    print('uint8 cache manifest:',CLIPS.video_id.nunique(),'source videos; training seeds:',CFG.TRAIN_SEEDS)


In [ ]:
# CELL 7 — reference-consistent model labels and temporal architecture implementations
# Model architectures are adapted from the uploaded original notebook.
class AttentionBiGRU(nn.Module):
    def __init__(self,dim,hidden=256):
        super().__init__()
        self.rnn=nn.GRU(dim,hidden,batch_first=True,bidirectional=True,num_layers=2,dropout=.3)
        self.attn=nn.Linear(hidden*2,1)
    def forward(self,x):
        o,_=self.rnn(x); weights=torch.softmax(self.attn(o),dim=1)
        return (weights*o).sum(1)

class EfficientNetBiGRU(nn.Module):
    def __init__(self, pretrained=None):
        super().__init__()
        pretrained = CFG.PRETRAINED if pretrained is None else pretrained
        net=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        self.encoder=net.features; self.pool=nn.AdaptiveAvgPool2d(1)
        self.temporal=AttentionBiGRU(net.classifier[1].in_features)
        self.head=nn.Sequential(nn.Dropout(.5),nn.Linear(512,2))
    def forward(self,x):
        b,t,c,h,w=x.shape
        feat=self.pool(self.encoder(x.reshape(b*t,c,h,w))).flatten(1).reshape(b,t,-1)
        return self.head(self.temporal(feat))

class MobileNetV3TSM(nn.Module):
    def __init__(self, pretrained=None):
        super().__init__()
        pretrained = CFG.PRETRAINED if pretrained is None else pretrained
        net=models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None)
        self.blocks=net.features; self.pool=nn.AdaptiveAvgPool2d(1)
        self.shift_before={3,7,12}; self.output_dim=net.classifier[0].in_features
        self.head=nn.Sequential(nn.Dropout(.5),nn.Linear(self.output_dim,2))
    @staticmethod
    def shift(x,b,t):
        bt,c,h,w=x.shape; assert bt==b*t
        x=x.reshape(b,t,c,h,w)
        fold=max(1,c//8)
        out=torch.zeros_like(x)
        out[:,1:,:fold]=x[:,:-1,:fold]
        out[:,:-1,fold:2*fold]=x[:,1:,fold:2*fold]
        out[:,:,2*fold:]=x[:,:,2*fold:]
        return out.reshape(bt,c,h,w)
    def forward(self,x):
        b,t,c,h,w=x.shape; x=x.reshape(b*t,c,h,w)
        for i,layer in enumerate(self.blocks):
            if i in self.shift_before: x=self.shift(x,b,t)
            x=layer(x)
        return self.head(self.pool(x).flatten(1).reshape(b,t,-1).mean(1))

class Compact3DBlock(nn.Module):
    def __init__(self,ci,co,stride=(1,2,2)):
        super().__init__()
        self.net=nn.Sequential(nn.Conv3d(ci,co,(3,1,1),padding=(1,0,0),bias=False),
            nn.BatchNorm3d(co),nn.ReLU(inplace=True),
            nn.Conv3d(co,co,(1,3,3),stride=stride,padding=(0,1,1),bias=False),
            nn.BatchNorm3d(co))
        self.skip=(nn.Sequential(nn.Conv3d(ci,co,1,stride=stride,bias=False),nn.BatchNorm3d(co))
                   if ci!=co or stride!=(1,1,1) else nn.Identity())
    def forward(self,x): return F.relu(self.net(x)+self.skip(x))

class Compact3DX3DInspired(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv3d(3,24,3,stride=(1,2,2),padding=1,bias=False),nn.BatchNorm3d(24),nn.ReLU(inplace=True))
        self.body=nn.Sequential(Compact3DBlock(24,48),Compact3DBlock(48,96),
                                Compact3DBlock(96,192),Compact3DBlock(192,432,stride=(1,1,1)))
        self.head=nn.Sequential(nn.AdaptiveAvgPool3d((1,1,1)),nn.Flatten(),nn.Linear(432,1024),
                                nn.GELU(),nn.Dropout(.5),nn.Linear(1024,2))
    def forward(self,x): return self.head(self.body(self.stem(x.permute(0,2,1,3,4).contiguous())))

REGISTRY={'EfficientNet-B0+BiGRU':EfficientNetBiGRU,
          'MobileNetV3+TSM':MobileNetV3TSM,
          'Compact3D-X3D-inspired':Compact3DX3DInspired}

def build_model(name, pretrained=None):
    if name not in REGISTRY: raise KeyError(name)
    cls = REGISTRY[name]
    return (cls() if name == 'Compact3D-X3D-inspired' else cls(pretrained=pretrained)).to(DEVICE)

print('Model classes:',list(REGISTRY),'(Compact3D is not canonical X3D-XS)')

In [ ]:
# CELL 8 — static-frame control, training utilities and checkpoint selection
import contextlib
class StaticFrameEfficientNet(nn.Module):
    def __init__(self, pretrained=None):
        super().__init__()
        pretrained = CFG.PRETRAINED if pretrained is None else pretrained
        net=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        net.classifier=nn.Sequential(nn.Dropout(.5),nn.Linear(net.classifier[1].in_features,2))
        self.net=net
    def forward(self,x): return self.net(x[:,-1])
REGISTRY['SingleFrame-EfficientNet'] = StaticFrameEfficientNet
STATIC_BASELINE='SingleFrame-EfficientNet'

class FocalLoss(nn.Module):
    def __init__(self,alpha=.25,gamma=2.,smoothing=.05):
        super().__init__(); self.alpha=alpha; self.gamma=gamma; self.smoothing=smoothing
    def forward(self,logits,y):
        ce=F.cross_entropy(logits,y,reduction='none',label_smoothing=self.smoothing)
        pt=torch.softmax(logits,dim=1).gather(1,y[:,None]).squeeze(1)
        alpha=torch.where(y==1,self.alpha,1-self.alpha)
        return (alpha*(1-pt).pow(self.gamma)*ce).mean()

class ModelEMA:
    def __init__(self,model,decay=.999):
        self.model=copy.deepcopy(model).eval()
        for p in self.model.parameters(): p.requires_grad_(False)
        self.decay=decay
    @torch.no_grad()
    def update(self,src):
        a=self.model.state_dict(); b=src.state_dict()
        for key in a:
            if torch.is_floating_point(a[key]): a[key].mul_(self.decay).add_(b[key],alpha=1-self.decay)
            else: a[key].copy_(b[key])

def maybe_sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

def amp_context():
    return torch.autocast(device_type='cuda',dtype=torch.float16) if (DEVICE.type=='cuda' and CFG.AMP_ON_CUDA) else contextlib.nullcontext()

@torch.no_grad()
def predict_probabilities(model,loader):
    model.eval(); probs=[]; ys=[]
    for clips,labels in loader:
        clips=preprocess_uint8_batch(clips,training=False)
        with amp_context(): out=model(clips)
        probs.extend(torch.softmax(out.float(),dim=1)[:,1].detach().cpu().numpy().tolist())
        ys.extend(labels.numpy().astype(int).tolist())
    return np.asarray(ys,dtype=int),np.asarray(probs,dtype=float)

def validate_regime(model,regime):
    horizons=('EC',) if regime=='EC' else CFG.HORIZONS
    aps=[]
    for h in horizons:
        dl=get_loader('tune',regime,None if h=='EC' else h)
        y,p=predict_probabilities(model,dl)
        if len(np.unique(y))<2: raise RuntimeError('Tuning has just one class')
        aps.append(float(average_precision_score(y,p)))
    return float(np.mean(aps)),dict(zip(map(clip_key,horizons),aps))

def checkpoint_path(regime, name):
    return RUN_ROOT/'checkpoints'/f'{model_key(regime, name)}.pth'

def training_state_path(regime, name):
    return RUN_ROOT/'checkpoints'/f'{model_key(regime, name)}.train_state.pth'

def training_complete_path(regime, name):
    return RUN_ROOT/'checkpoints'/f'{model_key(regime, name)}.complete.json'

def training_signature(regime, name):
    # Detect drift in the experiment definition before resuming or loading.
    keys = ('EPOCHS','EVAL_EVERY','FRAMES','IMAGE_SIZE','CLIP_SECONDS','HORIZONS',
            'LR','WEIGHT_DECAY','ACCUM_STEPS','TRAIN_BATCH_SIZE','PRETRAINED',
            'AMP_ON_CUDA','GPU_AUGMENT','STRICT_DETERMINISM','SEED')
    config = {key: getattr(CFG, key) for key in keys}
    config.update({'regime': regime, 'model': name, 'run_seed': RUN_SEED,
                   'split_sha256': sha256_bytes(ROOT/'manifests'/'split_manifest.csv'),
                   'clip_sha256': sha256_bytes(ROOT/'manifests'/'clip_manifest.csv')})
    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()

def save_torch_atomic(state, path):
    tmp = path.with_suffix('.part.pth')
    try:
        torch.save(state, tmp)
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)

def rng_snapshot():
    return {'python': random.getstate(), 'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else None}

def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    if DEVICE.type == 'cuda':
        if state['cuda'] is None:
            raise RuntimeError('Training state contains no CUDA RNG; resume on the original accelerator.')
        torch.cuda.set_rng_state_all(state['cuda'])

def assert_training_complete(regime, name):
    path = checkpoint_path(regime, name)
    marker = training_complete_path(regime, name)
    if not marker.is_file() or not path.is_file():
        raise RuntimeError(f'Training incomplete for {regime}/{name}; run or resume training before inference.')
    info = json.loads(marker.read_text())
    if (info.get('epochs_completed') != CFG.EPOCHS or
        info.get('training_signature') != training_signature(regime, name) or
        info.get('best_checkpoint_sha256') != sha256_bytes(path)):
        raise RuntimeError(f'Training completion marker is stale or does not match checkpoint: {path}')

def train_model(regime, name):
    path = checkpoint_path(regime, name)
    state_path = training_state_path(regime, name)
    completion = training_complete_path(regime, name)
    signature = training_signature(regime, name)
    if completion.exists():
        assert_training_complete(regime, name)
        print('Completed training verified:', path.name)
        return path
    if path.exists() and not state_path.exists():
        raise RuntimeError(f'Best checkpoint without resumable state: {path}. '
                           'Cannot treat interrupted legacy training as complete. Use a fresh WORK_DIR.')

    seed_all(RUN_SEED)
    model = build_model(name, pretrained=(CFG.PRETRAINED and not state_path.exists()))
    ema = ModelEMA(model)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = CosineAnnealingWarmRestarts(opt, T_0=7, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda' and CFG.AMP_ON_CUDA))
    loss_fn = FocalLoss(alpha=.25, gamma=2., smoothing=.05)
    dl = get_loader('train', regime, training=True, seed=RUN_SEED)
    hist, best, best_epoch, start_epoch, best_hash = [], -np.inf, -1, 0, None

    if state_path.exists():
        prev = torch.load(state_path, map_location='cpu', weights_only=False)
        if (prev.get('training_signature') != signature or prev.get('seed') != RUN_SEED or
            prev.get('device_type') != DEVICE.type):
            raise RuntimeError('Cannot resume: seed, split, or training configuration changed.')
        model.load_state_dict(prev['model'])
        ema.model.load_state_dict(prev['ema'])
        opt.load_state_dict(prev['optimizer'])
        scheduler.load_state_dict(prev['scheduler'])
        scaler.load_state_dict(prev['scaler'])
        hist = prev['history']
        best, best_epoch = prev['best'], prev['best_epoch']
        best_hash = prev['best_checkpoint_sha256']
        if not path.exists() or sha256_bytes(path) != best_hash:
            raise RuntimeError('Best checkpoint does not match the last completed epoch state; '
                               'interruption occurred during a checkpoint transaction. '
                               'Do not silently resume inconsistent model state.')
        start_epoch = int(prev['epoch_completed'])
        restore_rng(prev['rng'])
        print(f'Resuming {regime}/{name} after complete epoch {start_epoch}/{CFG.EPOCHS}')
    else:
        print(f'Training {regime}/{name} for {CFG.EPOCHS} epochs')

    for epoch in range(start_epoch, CFG.EPOCHS):
        dl.dataset.set_epoch(epoch)
        dl.sampler.generator.manual_seed(RUN_SEED + 1009 * epoch)
        model.train(); opt.zero_grad(set_to_none=True)
        seen = 0; total_loss = 0.0
        total_steps = len(dl)
        for step, (clips, labels) in enumerate(tqdm(dl, desc=f'{regime}/{clean_model_name(name)} {epoch+1}', leave=False), start=1):
            clips = preprocess_uint8_batch(clips, training=True)
            labels = labels.to(DEVICE, non_blocking=(DEVICE.type=='cuda'))
            with amp_context():
                logits = model(clips)
                loss = loss_fn(logits, labels)
            total_loss += float(loss.detach()) * len(labels)
            seen += len(labels)
            # Correct weighting of the last accumulation group when incomplete.
            group_start = ((step - 1) // CFG.ACCUM_STEPS) * CFG.ACCUM_STEPS
            group_size = min(CFG.ACCUM_STEPS, total_steps - group_start)
            scaler.scale(loss / group_size).backward()
            if step % CFG.ACCUM_STEPS == 0 or step == total_steps:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)
                ema.update(model)
        scheduler.step(epoch + 1)
        row = {'epoch': epoch + 1, 'train_loss': total_loss / max(1, seen)}
        if (epoch + 1) % CFG.EVAL_EVERY == 0 or epoch == CFG.EPOCHS - 1:
            metric, by_h = validate_regime(ema.model, regime)
            row['tuning_selection_AP'] = metric
            row.update({'AP_' + key: value for key, value in by_h.items()})
            print(f'  epoch {epoch+1}: tuning mean AP={metric:.4f}')
            if metric > best:
                best, best_epoch = metric, epoch + 1
                save_torch_atomic({'cache_backend': CFG.CACHE_BACKEND,
                                   'regime': regime, 'model_name': name,
                                   'state_dict': ema.model.state_dict(),
                                   'best_epoch': best_epoch, 'tune_selection_AP': best,
                                   'seed': RUN_SEED, 'training_signature': signature,
                                   'architecture_note': 'Custom X3D-inspired 3D CNN, not canonical X3D-XS'}, path)
                best_hash = sha256_bytes(path)
        hist.append(row)
        # Save once per COMPLETED epoch: interruption can safely restart at that boundary.
        save_torch_atomic({'training_signature': signature, 'seed': RUN_SEED,
                           'device_type': DEVICE.type, 'best_checkpoint_sha256': best_hash,
                           'epoch_completed': epoch+1, 'model': model.state_dict(),
                           'ema': ema.model.state_dict(), 'optimizer': opt.state_dict(),
                           'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                           'history': hist, 'best': best, 'best_epoch': best_epoch,
                           'rng': rng_snapshot()}, state_path)
        pd.DataFrame(hist).to_csv(RUN_ROOT/'logs'/f'history_{model_key(regime,name)}.csv', index=False)
    if not path.exists() or best_epoch < 0:
        raise RuntimeError(f'No tuning-selected checkpoint for {regime}/{name}')
    save_json({'epochs_completed': CFG.EPOCHS, 'training_signature': signature,
               'best_epoch': best_epoch, 'best_tune_AP': best,
               'best_checkpoint_sha256': sha256_bytes(path)}, completion)
    state_path.unlink(missing_ok=True)  # completed runs only need best weights, not full optimizer state
    del model, ema
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    print('Selected', path.name, 'best epoch', best_epoch, 'tune AP', best)
    return path

def load_model(regime, name):
    assert_training_complete(regime, name)
    path = checkpoint_path(regime, name)
    ck = torch.load(path, map_location='cpu', weights_only=False)
    if (ck.get('regime') != regime or ck.get('model_name') != name or
        ck.get('seed') != RUN_SEED or
        ck.get('training_signature') != training_signature(regime, name)):
        raise RuntimeError('Checkpoint model, seed, or training protocol mismatch')
    model = build_model(name, pretrained=False)  # No ImageNet download for inference.
    model.load_state_dict(ck['state_dict'], strict=True)
    model.eval()
    return model

if 'train' in CFG.STAGES:
    TRAIN_LIST=[('EC',m) for m in CFG.ENCODERS]+[('PE',m) for m in CFG.ENCODERS]+[('PE',STATIC_BASELINE)]
    for training_seed in CFG.RUN_SEEDS:
        activate_seed(training_seed)
        for regime,name in TRAIN_LIST: train_model(regime,name)


In [ ]:
# CELL 9 — deterministic tuning / final inference, indexed by original video ID
# The TEST stage is guarded in the selection cell. Never inspect final-test predictions during model selection.

def inference_path(split,regime,name,h):
    return RUN_ROOT/'predictions'/f'{split}__{model_key(regime,name)}__{clip_key(h)}.csv'

def infer_one(split,regime,name,h):
    path=inference_path(split,regime,name,h)
    if split == 'test' and (not final_test_gate_is_open() or not (RUN_ROOT/'selection'/'selection.lock.json').is_file()):
        raise PermissionError('Final-test predictions require the frozen selection and explicit unlock')
    assert_training_complete(regime, name)
    metadata_path = path.with_suffix('.meta.json')
    expected = {'checkpoint_sha256': sha256_bytes(checkpoint_path(regime, name)),
                'clip_manifest_sha256': sha256_bytes(ROOT/'manifests'/'clip_manifest.csv'),
                'split': split, 'regime': regime, 'name': name, 'horizon': clip_key(h)}
    if path.exists():
        if not metadata_path.exists():
            raise RuntimeError(f'Missing prediction provenance: {path}')
        recorded = json.loads(metadata_path.read_text())
        if any(recorded.get(k) != v for k,v in expected.items()) or recorded.get('prediction_sha256') != sha256_bytes(path):
            raise RuntimeError(f'Stale/incompatible prediction CSV: {path}; do not silently reuse it')
        return pd.read_csv(path, dtype={'video_id':str})
    model=load_model(regime,name)
    loader_regime='PE' if (regime=='EC' and h!='EC') else regime
    dl=get_loader(split,loader_regime,None if h=='EC' else h,training=False)
    y,p=predict_probabilities(model,dl)
    meta=dl.dataset.rows[['video_id','target','event_time','alert_time','clip_start','clip_end']].copy().reset_index(drop=True)
    if not np.array_equal(meta.target.to_numpy(int),y): raise RuntimeError('Predictions misaligned with manifest labels')
    meta['split']=split; meta['regime']=regime; meta['model_name']=name
    meta['horizon']=clip_key(h); meta['prob_collision']=p
    temp = path.with_suffix('.part.csv')
    meta.to_csv(temp, index=False)
    temp.replace(path)
    save_json({**expected, 'prediction_sha256': sha256_bytes(path)}, metadata_path)
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return meta

def infer_split(split):
    if split=='test' and not final_test_gate_is_open(): raise PermissionError('Final-test stage locked; verify all frozen seed selections first')
    specs=[('EC',m,('EC',)+CFG.HORIZONS) for m in CFG.ENCODERS]
    specs += [('PE',m,CFG.HORIZONS) for m in list(CFG.ENCODERS)+[STATIC_BASELINE]]
    for regime,model_name,horizons in specs:
        for h in horizons: infer_one(split,regime,model_name,h)
    print(f'{split} inference exported for {len(specs)} trained model/regime variants')

# Only produce tuning predictions before selection. Tests are generated later.
if 'select' in CFG.STAGES:
    for training_seed in CFG.RUN_SEEDS:
        activate_seed(training_seed)
        infer_split('tune')


In [ ]:
# CELL 10 — tuning-only fusion, 2-model ablations, calibration and thresholds
from scipy.optimize import minimize_scalar
from itertools import combinations

def load_predictions(split,regime,names,h):
    if split == 'test' and not (RUN_ROOT/'selection'/'final_test_completed.json').is_file():
        raise PermissionError('Cannot inspect test predictions before final inference is completed and sealed')
    dfs=[pd.read_csv(inference_path(split,regime,name,h),dtype={'video_id':str}) for name in names]
    joined=dfs[0][['video_id','target','event_time','alert_time']].rename(columns={'target':'label'}).copy()
    joined['video_id']=joined.video_id.astype(str)
    cols=[]
    for i,d in enumerate(dfs):
        source=d[['video_id','target','prob_collision']].rename(columns={'target':f'lab_{i}','prob_collision':f'p_{i}'})
        joined=joined.merge(source,on='video_id',how='inner',validate='one_to_one')
        if len(joined)!=len(dfs[0]) or not (joined[f'lab_{i}']==joined.label).all():
            raise RuntimeError('Model predictions are missing/misaligned; refuse positional array fusion')
        cols.append(f'p_{i}')
    assert joined.video_id.is_unique
    return joined,np.column_stack([joined[c].to_numpy(float) for c in cols])

def prepare_fusion_arrays(split, regime, names):
    return [(df.label.to_numpy(int), p) for h in (('EC',) if regime=='EC' else CFG.HORIZONS)
            for df, p in [load_predictions(split, regime, names, h)]]

def horizon_metric(split, regime, names, weights, metric='ap', prepared=None):
    arrays = prepared if prepared is not None else prepare_fusion_arrays(split, regime, names)
    w = np.asarray(weights, dtype=float)
    scores = [float(average_precision_score(y, p@w)) if metric == 'ap'
              else float(roc_auc_score(y, p@w)) for y, p in arrays]
    return float(np.mean(scores)), scores

def grid_weights(n,step):
    count=int(round(1./step))
    if not np.isclose(count*step,1.): raise ValueError('Fusion step must divide 1')
    if n==1: yield np.array([1.]); return
    if n==2:
        for i in range(count+1): yield np.array([i,count-i],float)/count
    elif n==3:
        for i in range(count+1):
            for j in range(count-i+1): yield np.array([i,j,count-i-j],float)/count
    else: raise NotImplementedError('Expected 2 or 3 branches')

def search_fusion(names):
    prepared=prepare_fusion_arrays('tune', 'PE', names)
    best=None
    for weights in grid_weights(len(names),CFG.FUSION_GRID_STEP):
        # True three-branch ablation: enforce every candidate branch has at least 5% weight.
        if len(names)==3 and min(weights)<0.05-1e-12: continue
        # Tuning-only. Deterministic tie handling: first candidate wins.
        score,_=horizon_metric('tune','PE',names,weights,prepared=prepared)
        if best is None or score>best['tune_hmAP']+1e-12:
            best={'names':list(names),'weights':weights.tolist(),'tune_hmAP':score}
    return best

def optimal_fpr_threshold(y,p,limit):
    y=np.asarray(y,dtype=int); p=np.asarray(p,dtype=float)
    candidates=np.unique(np.r_[0.,1.,np.nextafter(1.,2.),p])
    candidates.sort()
    best=None
    for t in candidates:
        pred=p>=t
        fpr=float(np.mean(pred[y==0]))
        tpr=float(np.mean(pred[y==1]))
        if fpr<=limit+1e-12:
            if best is None or tpr>best['tpr']+1e-12 or (abs(tpr-best['tpr'])<1e-12 and fpr<best['fpr']):
                best={'threshold':float(t),'tpr':tpr,'fpr':fpr}
    if best is None: raise RuntimeError('Could not find feasible FPR-constrained operating threshold')
    return best

def youden_threshold(y,p):
    cand=np.unique(np.r_[0.,1.,np.nextafter(1.,2.),p]); best=None
    for t in cand:
        pred=p>=t
        se=float(np.mean(pred[y==1])); sp=float(np.mean(~pred[y==0])); j=se+sp-1
        if best is None or j>best['J']+1e-12:
            best={'threshold':float(t),'J':j,'sensitivity':se,'specificity':sp}
    return best

def pooled_predictions(split,names,weights,regime='PE'):
    frames=[]
    for h in (('EC',) if regime=='EC' else CFG.HORIZONS):
        df,p=load_predictions(split,regime,names,h)
        df=df[['video_id','label','event_time','alert_time']].copy()
        df['horizon']=clip_key(h); df['score']=p@np.array(weights,dtype=float)
        frames.append(df)
    return pd.concat(frames,ignore_index=True)

def tuning_temperature(y,p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    logit=np.log(p/(1-p)); y=np.asarray(y,int)
    def obj(log_t):
        return log_loss(y,expit(logit/np.exp(log_t)),labels=[0,1])
    res=minimize_scalar(obj,bounds=(np.log(.3),np.log(8.)),method='bounded')
    return float(np.exp(res.x))

def apply_temperature(p,T):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return expit(np.log(p/(1-p))/T)

def verify_frozen_lock_inputs(sel):
    for field, path in [('split_sha256', ROOT/'manifests'/'split_manifest.csv'),
                        ('clip_manifest_sha256', ROOT/'manifests'/'clip_manifest.csv')]:
        if sha256_bytes(path) != sel[field]:
            raise RuntimeError(f'Frozen tuning selection is incompatible: {field}')
    for category in ('tune_prediction_sha256','checkpoint_sha256'):
        for rel, digest in sel[category].items():
            if sha256_bytes(ROOT/rel) != digest:
                raise RuntimeError(f'Frozen tuning input changed: {rel}')
    if sel['run_seed'] != RUN_SEED or tuple(sel['all_predeclared_seeds']) != CFG.TRAIN_SEEDS:
        raise RuntimeError('Frozen seed mismatch')

def create_tuning_lock():
    path=RUN_ROOT/'selection'/'selection.lock.json'
    if path.exists():
        print('Using existing frozen selection:',path)
        existing = json.loads(path.read_text())
        verify_frozen_lock_inputs(existing)
        return existing
    best_single=None
    for name in CFG.ENCODERS:
        s,_=horizon_metric('tune','PE',[name],[1.])
        if best_single is None or s>best_single['tune_hmAP']:
            best_single={'names':[name],'weights':[1.0],'tune_hmAP':s}
    three_names=list(CFG.ENCODERS)
    best_two=max((search_fusion(pair) for pair in combinations(three_names,2)),key=lambda d:d['tune_hmAP'])
    equal2={'names':best_two['names'],'weights':[.5,.5]}
    equal3={'names':three_names,'weights':[1/3]*3}
    best_three=search_fusion(three_names)
    # The primary candidate is preselected as optimal three-way fusion; F2 vs F3 is secondary.
    primary=best_three
    pooled=pooled_predictions('tune',primary['names'],primary['weights'])
    common=optimal_fpr_threshold(pooled.label,pooled.score,CFG.FPR_TUNING_TARGET)
    horizon_rules={}
    for h in CFG.HORIZONS:
        d=pooled[pooled.horizon==clip_key(h)]
        horizon_rules[clip_key(h)]=optimal_fpr_threshold(d.label,d.score,CFG.FPR_TUNING_TARGET)
    T=tuning_temperature(pooled.label,pooled.score)
    # EC threshold-transfer diagnostic uses EC-tuned model, not the PE model/threshold.
    ec_opt=search_ec_fusion()
    ec_pooled=pooled_predictions('tune',ec_opt['names'],ec_opt['weights'],regime='EC')
    ec_youden=youden_threshold(ec_pooled.label,ec_pooled.score)
    pred_paths=sorted((RUN_ROOT/'predictions').glob('tune__*.csv'))
    hashes={str(p.relative_to(ROOT)):sha256_bytes(p) for p in pred_paths}
    checkpoints={str(checkpoint_path(r,m).relative_to(ROOT)):sha256_bytes(checkpoint_path(r,m))
                 for r,m in ([('EC',z) for z in CFG.ENCODERS]+[('PE',z) for z in list(CFG.ENCODERS)+[STATIC_BASELINE]])}
    result={'protocol':'NEXAR_TVDR_V2_MULTI_SEED_UINT8','run_seed':RUN_SEED,'all_predeclared_seeds':list(CFG.TRAIN_SEEDS),'cache_backend':CFG.CACHE_BACKEND,'created_at_utc':datetime.now(timezone.utc).isoformat(),
            'pilot_only':CFG.DRY_RUN,
            'split_sha256':sha256_bytes(ROOT/'manifests'/'split_manifest.csv'),
            'clip_manifest_sha256':sha256_bytes(ROOT/'manifests'/'clip_manifest.csv'),
            'tune_prediction_sha256':hashes,'checkpoint_sha256':checkpoints,
            'selection_endpoint':'tune_horizon_mean_AP',
            'h1_primary_model':best_single['names'][0],
            'best_single':best_single,'best_two':best_two,'equal_two':equal2,
            'equal_three':equal3,'best_three':best_three,'primary_fusion':primary,
            'threshold_common':common,'threshold_horizon':horizon_rules,
            'temperature':T,
            'ec_diagnostic_fusion':ec_opt,'ec_diagnostic_youden':ec_youden,
            'FPR_TUNING_TARGET':CFG.FPR_TUNING_TARGET,
            'NOTE':'All model, fusion, temperature, thresholds chosen on tune. Test untouched.'}
    save_json(result,path)
    print('FROZEN tuning selection:',path)
    print('Best PE single:',best_single,'Best two:',best_two,'Best three:',best_three)
    return result

def search_ec_fusion():
    prepared=prepare_fusion_arrays('tune','EC',list(CFG.ENCODERS))
    best=None
    for weights in grid_weights(len(CFG.ENCODERS),CFG.FUSION_GRID_STEP):
        if min(weights)<0.05-1e-12: continue
        score,_=horizon_metric('tune','EC',list(CFG.ENCODERS),weights,prepared=prepared)
        if best is None or score>best['tune_hmAP']+1e-12:
            best={'names':list(CFG.ENCODERS),'weights':weights.tolist(),'tune_hmAP':score}
    return best

SELECTION=None
if 'select' in CFG.STAGES:
    for training_seed in CFG.RUN_SEEDS:
        activate_seed(training_seed)
        SELECTION=create_tuning_lock()
elif (RUN_ROOT/'selection'/'selection.lock.json').exists():
    SELECTION=json.loads((RUN_ROOT/'selection'/'selection.lock.json').read_text())


In [ ]:
# CELL 11 — FINAL TEST GATE: must be enabled only after tuning selections are frozen
# Full source videos were used for earlier development; old checkpoints must NOT be imported.

def confirm_frozen_selection(sel):
    if sel is None:
        raise RuntimeError('No frozen selection: execute select stage first')
    verify_frozen_lock_inputs(sel)
    if sel['pilot_only'] != CFG.DRY_RUN:
        raise RuntimeError('DRY_RUN mismatch versus frozen selection')
    if sel.get('cache_backend') != CFG.CACHE_BACKEND:
        raise RuntimeError('Cache backend changed after lock')
    return True

def require_all_seed_locks():
    missing=[]
    for s in CFG.TRAIN_SEEDS:
        lock=ROOT/'runs'/f'seed_{s}'/'selection'/'selection.lock.json'
        if not lock.exists(): missing.append(str(s))
    if missing: raise RuntimeError('Cannot unlock test until ALL prespecified seed locks exist: missing '+','.join(missing))

# One-click capability is in-memory and is ONLY granted after every selection lock
# passes content-hash verification. Rerunning the notebook resets the gate.
_ONE_CLICK_LOCK_DIGESTS = {}

def final_test_gate_is_open():
    if CFG.EXECUTION_MODE == 'staged':
        return bool(CFG.FINAL_TEST_UNLOCK)
    if len(_ONE_CLICK_LOCK_DIGESTS) != len(CFG.TRAIN_SEEDS):
        return False
    if set(_ONE_CLICK_LOCK_DIGESTS) != set(CFG.TRAIN_SEEDS):
        return False
    # If ANY predeclared seed's lock changes after the gate opens, fail closed.
    for seed,digest in _ONE_CLICK_LOCK_DIGESTS.items():
        path=ROOT/'runs'/f'seed_{seed}'/'selection'/'selection.lock.json'
        if not path.is_file() or sha256_bytes(path)!=digest:
            return False
    return True

def seal_and_open_one_click_gate():
    global _ONE_CLICK_LOCK_DIGESTS
    _ONE_CLICK_LOCK_DIGESTS = {}  # fail closed during verification
    require_all_seed_locks()
    verified = {}
    for seed in CFG.TRAIN_SEEDS:
        activate_seed(seed)
        path=RUN_ROOT/'selection'/'selection.lock.json'
        sel=json.loads(path.read_text())
        confirm_frozen_selection(sel)
        verified[seed] = sha256_bytes(path)
    # Publish the gate atomically ONLY after all three locks are validated.
    _ONE_CLICK_LOCK_DIGESTS=verified
    print('AUTO FINAL TEST GATE: unlocked only for frozen, verified selections of all seeds')
    return verified

if 'test' in CFG.STAGES:
    require_all_seed_locks()
    for training_seed in CFG.RUN_SEEDS:
        activate_seed(training_seed)
        SELECTION=json.loads((RUN_ROOT/'selection'/'selection.lock.json').read_text())
        confirm_frozen_selection(SELECTION)
        if not CFG.FINAL_TEST_UNLOCK:
            raise PermissionError('FINAL TEST IS LOCKED. Set FINAL_TEST_UNLOCK=True explicitly after checking selection.lock.json; do not revise selected models after inspecting test.')
        completion=RUN_ROOT/'selection'/'final_test_completed.json'
        if completion.exists() and not CFG.ALLOW_OVERWRITE_FINAL:
            raise RuntimeError('Final test already completed. Read its existing results; do not rerun/reselect based on outcomes.')
        infer_split('test')
        save_json({'final_inference_completed_utc':datetime.now(timezone.utc).isoformat(),
                   'selection_sha256':sha256_bytes(RUN_ROOT/'selection'/'selection.lock.json'),
                   'pilot_only':CFG.DRY_RUN},completion)
        print('Final test inference complete. No further model selection permitted.')

In [ ]:
# CELL 12 — statistical tests: preserve video-level repeated-horizon dependence
from statsmodels.stats.multitest import multipletests

def metric_value(y,p,what):
    if what=='AP': return float(average_precision_score(y,p))
    if what=='AUROC': return float(roc_auc_score(y,p))
    if what=='Brier': return float(brier_score_loss(y,p))
    raise KeyError(what)

def binary_stats(y,p,threshold):
    y=np.asarray(y,int); p=np.asarray(p,float); pred=p>=threshold
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'n':int(len(y)),'AP':metric_value(y,p,'AP'),'AUROC':metric_value(y,p,'AUROC'),
            'Brier':metric_value(y,p,'Brier'),
            'F1':float(f1_score(y,pred,zero_division=0)),
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'MCC':float(matthews_corrcoef(y,pred)),
            'sensitivity':float(tp/(tp+fn)) if tp+fn else np.nan,
            'FPR':float(fp/(tn+fp)) if tn+fp else np.nan,
            'FP':int(fp),'FN':int(fn),'TP':int(tp),'TN':int(tn)}

def ece(y,p,n_bins=10):
    y=np.asarray(y,int);p=np.asarray(p,float)
    edges=np.linspace(0,1,n_bins+1)
    ids=np.minimum(np.searchsorted(edges,p,side='right')-1,n_bins-1)
    ids=np.maximum(ids,0)
    return float(sum((np.sum(ids==i)/len(y))*abs(y[ids==i].mean()-p[ids==i].mean())
                     for i in range(n_bins) if np.any(ids==i)))

def as_horizon_arrays(split,names,weights):
    data=[]
    for h in CFG.HORIZONS:
        meta,p=load_predictions(split,'PE',names,h)
        d=meta[['video_id','label']].copy(); d['score']=p@np.asarray(weights,float)
        data.append(d.sort_values('video_id').reset_index(drop=True))
    base=data[0]
    for d in data[1:]:
        if not base.video_id.equals(d.video_id) or not base.label.equals(d.label):
            raise RuntimeError('Unpaired horizons. Cannot run paired inference.')
    return base.video_id.to_numpy(),base.label.to_numpy(int),np.column_stack([d.score.to_numpy(float) for d in data])

def hmAP(y,P): return float(np.mean([average_precision_score(y,P[:,i]) for i in range(P.shape[1])]))

def stratified_bootstrap_video_indices(y,rng):
    a=np.flatnonzero(y==0);b=np.flatnonzero(y==1)
    return np.concatenate([rng.choice(a,len(a),replace=True),rng.choice(b,len(b),replace=True)])

def paired_inference(y,A,B,statfn,n_boot=2000,n_perm=1999,seed=42):
    y=np.asarray(y,int);A=np.asarray(A,float);B=np.asarray(B,float)
    assert A.shape==B.shape and A.shape[0]==len(y)
    obs=float(statfn(y,A)-statfn(y,B))
    rng=np.random.default_rng(seed); samples=[]
    for _ in range(n_boot):
        ii=stratified_bootstrap_video_indices(y,rng)
        samples.append(float(statfn(y[ii],A[ii])-statfn(y[ii],B[ii])))
    interval=np.quantile(samples,[.025,.975])
    extreme=0
    for _ in range(n_perm):
        swap=(rng.integers(0,2,size=len(y))==1)[:,None]
        pa=np.where(swap,B,A);pb=np.where(swap,A,B)
        if abs(statfn(y,pa)-statfn(y,pb))>=abs(obs)-1e-12: extreme+=1
    p=(extreme+1)/(n_perm+1)
    return {'difference':obs,'CI_lower':float(interval[0]),'CI_upper':float(interval[1]),
            'p_perm_two_sided':float(p),'bootstraps':n_boot,'permutations':n_perm}

def two_confirmatory_tests(sel):
    # H1 compares time horizons for preselected best single; treat horizon columns as paired conditions.
    ids,y,single=as_horizon_arrays('test',sel['best_single']['names'],[1.])
    assert single.shape[1]==3
    h1=paired_inference(y,single[:,[0]],single[:,[2]],hmAP,
                         CFG.BOOTSTRAPS,CFG.PERMUTATIONS,CFG.SEED+10)
    # H2 compares a preselected 3-branch ensemble against tuning-best single at all three horizons.
    ids2,y2,ensemble=as_horizon_arrays('test',sel['primary_fusion']['names'],sel['primary_fusion']['weights'])
    if not np.array_equal(ids,ids2) or not np.array_equal(y,y2): raise RuntimeError('Non-paired model comparison')
    h2=paired_inference(y,ensemble,single,hmAP,CFG.BOOTSTRAPS,CFG.PERMUTATIONS,CFG.SEED+20)
    adjusted=multipletests([h1['p_perm_two_sided'],h2['p_perm_two_sided']],alpha=.05,method='holm')[1]
    for k,d,padj in [('H1',h1,adjusted[0]),('H2',h2,adjusted[1])]:
        d['hypothesis']=k;d['p_holm_two_sided']=float(padj)
    return pd.DataFrame([h1,h2]), {'single':single,'fusion':ensemble,'labels':y,'video_ids':ids}

# Unit sanity: no test data used here.
assert abs(ece([0,1],[.1,.9],10)-.1)<1e-8
assert abs(hmAP(np.array([0,1,0,1]),np.array([[.1,.2],[.8,.7],[.2,.1],[.9,.9]]))-1.)<1e-8
print('Paired bootstrap + horizon-aware permutation utilities ready.')

In [ ]:
# CELL 13 — final test tables, calibration, threshold transfer, cost and failure cases

def main_tables(sel):
    confirm_frozen_selection(sel)
    if not (RUN_ROOT/'selection'/'final_test_completed.json').exists():
        raise RuntimeError('Final test has not been executed. No empty or hypothetical paper tables are produced.')
    # Table I: actual cohort, not assumed 900/300/300 after exclusions.
    count=MANIFEST.groupby(['split','target']).size().unstack(fill_value=0)
    count.columns=['normal' if x==0 else 'critical_event' for x in count.columns]
    count['total']=count.sum(1)
    count.to_csv(RUN_ROOT/'tables'/'table_I_cohort.csv')
    plans=[('Single-frame',{'names':[STATIC_BASELINE],'weights':[1]}),
           ('EfficientNet + BiGRU',{'names':[CFG.ENCODERS[0]],'weights':[1]}),
           ('MobileNetV3 + TSM',{'names':[CFG.ENCODERS[1]],'weights':[1]}),
           ('Compact3D (X3D-inspired)',{'names':[CFG.ENCODERS[2]],'weights':[1]}),
           ('Equal 2-branch',sel['equal_two']),('Optimized 2-branch',sel['best_two']),
           ('Equal 3-branch',sel['equal_three']),('Optimized 3-branch',sel['best_three'])]
    paper=[]
    for label,item in plans:
        _,y,p=as_horizon_arrays('test',item['names'],item['weights'])
        scores=[average_precision_score(y,p[:,i]) for i in range(3)]
        paper.append({'Model':label,'AP_0.5':scores[0],'AP_1.0':scores[1],
                      'AP_1.5':scores[2],'h_mAP':float(np.mean(scores)),
                      'Brier_pooled':brier_score_loss(np.tile(y,3),np.hstack([p[:,j] for j in range(3)])),
                      'ECE10_pooled':ece(np.tile(y,3),np.hstack([p[:,j] for j in range(3)]))})
    table=pd.DataFrame(paper).sort_values('h_mAP',ascending=False)
    table.to_csv(RUN_ROOT/'tables'/'table_II_pre_event_models.csv',index=False)
    # Exploratory EC-trained models on strictly PE frames: out-of-distribution behavior,
    # never interpreted as a causal event-visibility contrast.
    ecrows=[];ec=sel['ec_diagnostic_fusion']
    for h in ('EC',)+CFG.HORIZONS:
        d,p=load_predictions('test','EC',ec['names'],h)
        q=p@np.asarray(ec['weights'],float)
        er=binary_stats(d.label.to_numpy(int),q,sel['ec_diagnostic_youden']['threshold'])
        er.update({'horizon':clip_key(h),'trained_on':'EC','evaluated_on':clip_key(h),
                   'threshold_origin':'EC_tune_youden'})
        ecrows.append(er)
    pd.DataFrame(ecrows).to_csv(RUN_ROOT/'tables'/'table_S_EC_to_PE_domain_shift.csv',index=False)
    stats,arr=two_confirmatory_tests(sel)
    stats.to_csv(RUN_ROOT/'stats'/'H1_H2_paired_results.csv',index=False)
    pooled=pooled_predictions('test',sel['primary_fusion']['names'],sel['primary_fusion']['weights'])
    scores=[]
    for h in CFG.HORIZONS:
        group=pooled[pooled.horizon==clip_key(h)].copy()
        y=group.label.to_numpy(int);p=group.score.to_numpy(float)
        for rule,threshold in [('common',sel['threshold_common']['threshold']),
                               ('horizon_specific',sel['threshold_horizon'][clip_key(h)]['threshold'])]:
            r=binary_stats(y,p,threshold);r.update({'horizon':clip_key(h),'threshold_rule':rule,'threshold':threshold})
            for ratio in (1,5,10):
                for prevalence in (.5,.05,.01):
                    r[f'expected_cost_ratio{ratio}_pi{prevalence}']=prevalence*ratio*(1-r['sensitivity'])+(1-prevalence)*r['FPR']
            scores.append(r)
    pd.DataFrame(scores).to_csv(RUN_ROOT/'tables'/'table_S_threshold_transfer.csv',index=False)
    before=pooled.score.to_numpy(float)
    after=apply_temperature(before,sel['temperature']); y=pooled.label.to_numpy(int)
    pd.DataFrame([{'mode':'uncalibrated','Brier':brier_score_loss(y,before),'NLL':log_loss(y,before),'ECE_10':ece(y,before)},
                  {'mode':'temperature_scaled','Brier':brier_score_loss(y,after),'NLL':log_loss(y,after),'ECE_10':ece(y,after)}]).to_csv(
        RUN_ROOT/'tables'/'table_S_calibration.csv',index=False)
    # Alert-relative diagnostic (positive subset only); anchor metadata never used for model input.
    diag=pooled[(pooled.label==1)&(pooled.alert_time.notna())].copy()
    if not diag.empty:
        hs={'H0.5':.5,'H1.0':1.,'H1.5':1.5}
        diag['cutoff_relative_to_alert']=diag.apply(lambda r:float(r.event_time)-hs[str(r.horizon)]-float(r.alert_time),axis=1)
        diag['observation_ends_before_alert']=diag.cutoff_relative_to_alert<0
        diag.to_csv(RUN_ROOT/'tables'/'table_S_alert_relative_positive_predictions.csv',index=False)
    # Error examples at one locked threshold. Do not generate explanations automatically.
    er=pooled[pooled.horizon=='H0.5'].copy()
    er['prediction']=(er.score>=sel['threshold_common']['threshold']).astype(int)
    er['error_kind']=np.select([(er.label==1)&(er.prediction==0),(er.label==0)&(er.prediction==1)],['FN','FP'],default='correct')
    er.sort_values(['error_kind','score']).to_csv(RUN_ROOT/'tables'/'table_S_error_cases.csv',index=False)
    save_json({'primary_hypotheses':stats.to_dict(orient='records'),'note':'All scores from locked independent test predictions.',
               'pilot_only':CFG.DRY_RUN},RUN_ROOT/'stats'/'result_summary.json')
    return table,stats

if 'report' in CFG.STAGES:
    if MANIFEST is None: MANIFEST=pd.read_csv(ROOT/'manifests'/'split_manifest.csv',dtype={'video_id':str})
    for training_seed in CFG.RUN_SEEDS:
        activate_seed(training_seed)
        SELECTION=json.loads((RUN_ROOT/'selection'/'selection.lock.json').read_text())
        FINAL_TABLE,CONFIRMATORY=main_tables(SELECTION)
        print('Seed',training_seed);display(FINAL_TABLE);display(CONFIRMATORY)


In [ ]:
# CELL 14 — manuscript Figure 1: reproducible temporal windows + data flow
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
plt.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','Nimbus Roman','Liberation Serif','DejaVu Serif'],
                     'font.size':8,'axes.titlesize':9,'axes.labelsize':8,'legend.fontsize':7,
                     'pdf.fonttype':42,'ps.fonttype':42,'savefig.dpi':300})
COLORS={'EC':'#aa7440','PE':'#237d86','threshold':'#80639c','single':'#3f72a7','three':'#864c96',
        'two':'#d18b32'}

def save_fig(fig,stem,shared=False):
    p=(ROOT if shared else RUN_ROOT)/'figures'/stem
    fig.savefig(p.with_suffix('.pdf'),bbox_inches='tight',facecolor='white')
    fig.savefig(p.with_suffix('.png'),bbox_inches='tight',dpi=300,facecolor='white')
    plt.close(fig)
    print('Saved:',p.with_suffix('.pdf').name,'and PNG')

def draw_protocol_figure():
    fig=plt.figure(figsize=(7.1,2.65));ax=fig.add_axes([.27,.12,.7,.82]);ax.set_xlim(-4.8,2.2);ax.set_ylim(-.3,5.1)
    ax.axvline(0,color='#b34841',lw=1.3,ls='--')
    for j,(lab,start,end,color) in enumerate([('Event-centered',-1.5,1.5,COLORS['EC']),
                                              ('Pre-event 0.5 s',-3.5,-.5,COLORS['PE']),
                                              ('Pre-event 1.0 s',-4.,-1.,COLORS['PE']),
                                              ('Pre-event 1.5 s',-4.5,-1.5,COLORS['PE'])]):
        y=4-j
        ax.add_patch(FancyBboxPatch((start,y-.30),end-start,.60,boxstyle='round,pad=.04',
                     facecolor=color,edgecolor=color,alpha=.19,lw=1.))
        ax.plot([start,end],[y,y],color=color,lw=1.5,alpha=.7)
        # Labels are y ticks outside the data region; they do not obscure the bars.
        if j==0: ax.axvspan(0,1.5,ymin=(y-.27+.3)/5.4,ymax=(y+.27+.3)/5.4,color='#b34841',alpha=.10)
    ax.text(.05,4.74,'Event time',color='#b34841',fontsize=8)
    ax.set_xticks([-4.5,-4,-3,-2,-1.5,-1,-.5,0,1,1.5]);ax.set_xlabel('Time relative to annotated event (seconds)')
    ax.set_yticks([4,3,2,1],['Event-centered','Pre-event 0.5 s','Pre-event 1.0 s','Pre-event 1.5 s']);ax.tick_params(axis='y',length=0)
    ax.spines[['left','right','top']].set_visible(False)
    ax.set_title('Strictly pre-event versus event-centered observation',fontweight='bold',loc='left')
    save_fig(fig,'fig_1_temporal_protocol',shared=True)

draw_protocol_figure()

In [ ]:
# CELL 15 — Figure 2: the MAIN scientific result (real independent-test data only)
def ci_for_single_ap(y,p,n_boot=500,seed=42):
    rng=np.random.default_rng(seed);values=[]
    for _ in range(n_boot):
        ii=stratified_bootstrap_video_indices(y,rng)
        values.append(average_precision_score(y[ii],p[ii]))
    return tuple(np.quantile(values,[.025,.975]))

def figure_2_horizons_and_fusion(sel):
    options=[('Single frame',[STATIC_BASELINE],[1.],'#708294'),
             ('EfficientNet + BiGRU',[CFG.ENCODERS[0]],[1.],'#3f72a7'),
             ('MobileNetV3 + TSM',[CFG.ENCODERS[1]],[1.],'#308d73'),
             ('Compact3D',[CFG.ENCODERS[2]],[1.],'#c16d4c'),
             ('Optimized 2-branch',sel['best_two']['names'],sel['best_two']['weights'],COLORS['two']),
             ('Optimized 3-branch',sel['best_three']['names'],sel['best_three']['weights'],COLORS['three'])]
    fig,axs=plt.subplots(1,2,figsize=(7.15,2.85),gridspec_kw={'width_ratios':[1.24,1]})
    for j,(label,names,weights,color) in enumerate(options):
        _,y,P=as_horizon_arrays('test',names,weights)
        means=[average_precision_score(y,P[:,i]) for i in range(3)]
        ci=np.array([ci_for_single_ap(y,P[:,i],seed=CFG.SEED+30+j*3+i) for i in range(3)])
        axs[0].errorbar(CFG.HORIZONS,means,yerr=np.vstack([means-ci[:,0],ci[:,1]-means]),
            marker='o',capsize=2.4,lw=1.25,ms=3.4,label=label,color=color,
            linestyle='--' if label=='Single frame' else '-')
    axs[0].set(xlabel='Pre-event horizon (s)',ylabel='Average precision',title='(a) Discrimination by temporal horizon')
    axs[0].set_xticks(CFG.HORIZONS);axs[0].set_ylim(0,1.03);axs[0].grid(alpha=.15)
    axs[0].legend(frameon=False,fontsize=6,loc='lower left')
    single=sel['best_single']; specs=[('Two-branch',sel['best_two']),('Three-branch',sel['best_three'])]
    _,y,p_single=as_horizon_arrays('test',single['names'],single['weights'])
    diffs=[]; lo=[]; hi=[]
    for j,(label,item) in enumerate(specs):
        _,_,p=as_horizon_arrays('test',item['names'],item['weights'])
        r=paired_inference(y,p,p_single,hmAP,n_boot=500,n_perm=499,seed=CFG.SEED+100+j)
        diffs.append(r['difference']);lo.append(r['CI_lower']);hi.append(r['CI_upper'])
    xs=np.arange(len(specs))
    axs[1].axvline(0,color='#444',lw=1,ls='--')
    for i in range(len(specs)):
        axs[1].errorbar(diffs[i],i,xerr=[[diffs[i]-lo[i]],[hi[i]-diffs[i]]],fmt='o',
                        color=[COLORS['two'],COLORS['three']][i],capsize=3)
    axs[1].set_yticks(xs,[x[0] for x in specs]);axs[1].invert_yaxis()
    axs[1].set(xlabel='Paired Δ horizon-mean AP (95% CI)',title='(b) Fusion over tuning-best single')
    axs[1].grid(alpha=.15,axis='x')
    fig.tight_layout(pad=1.)
    save_fig(fig,'fig_2_horizon_performance')

In [ ]:
# CELL 16 — Figure 3: common vs horizon-specific locked operating thresholds
# Report achieved test FPR (not assumed operational FPR target).
def figure_3_threshold_transfer():
    d=pd.read_csv(RUN_ROOT/'tables'/'table_S_threshold_transfer.csv')
    fig,axs=plt.subplots(1,2,figsize=(7.1,2.6))
    from statsmodels.stats.proportion import proportion_confint
    for irule,(rule,col,label) in enumerate([('common',COLORS['PE'],'Common PE threshold'),
                           ('horizon_specific',COLORS['threshold'],'Horizon-specific thresholds')]):
        sub=d[d.threshold_rule==rule].set_index('horizon')
        x=np.asarray(CFG.HORIZONS,float)+(irule-.5)*.025
        for j,(num,den,metric,mark) in enumerate([('TP','positive_total','sensitivity','o'),
                                                   ('FP','negative_total','FPR','s')]):
            vals=[]; lowers=[]; uppers=[]
            for h in CFG.HORIZONS:
                r=sub.loc[clip_key(h)]
                k=int(r[num]); n=int(r['TP']+r['FN']) if j==0 else int(r['TN']+r['FP'])
                lo,hi=proportion_confint(k,n,alpha=.05,method='wilson')
                vals.append(float(r[metric]));lowers.append(float(lo));uppers.append(float(hi))
            vals=np.asarray(vals)
            axs[j].errorbar(x,vals,yerr=[vals-np.asarray(lowers),np.asarray(uppers)-vals],
                            marker=mark,lw=1.35,ms=3.5,capsize=2,color=col,label=label)
    axs[1].axhline(CFG.FPR_TUNING_TARGET,color='gray',ls=':',lw=1.1,label='Tuning FPR target')
    for ax in axs:
        ax.set_xticks(CFG.HORIZONS);ax.set_xlim(.4,1.6);ax.set_ylim(-.025,1.025)
        ax.set_xlabel('Pre-event horizon (s)');ax.grid(alpha=.16)
    axs[0].set_ylabel('Sensitivity');axs[1].set_ylabel('False-positive rate')
    axs[0].set_title('(a) Sensitivity at locked thresholds')
    axs[1].set_title('(b) Achieved false-positive rate')
    handles,labels=axs[1].get_legend_handles_labels()
    fig.legend(handles,labels,frameon=False,loc='lower center',ncol=3,bbox_to_anchor=(.5,-.12),fontsize=7)
    fig.tight_layout(pad=1.1)
    save_fig(fig,'fig_3_threshold_transfer')

def figure_S_calibration(sel):
    d=pooled_predictions('test',sel['primary_fusion']['names'],sel['primary_fusion']['weights'])
    y=d.label.to_numpy(int);p=d.score.to_numpy(float)
    fig,ax=plt.subplots(figsize=(3.7,3.15));ax.plot([0,1],[0,1],ls='--',lw=1,color='gray',label='Ideal')
    for name,q,col in [('Uncalibrated',p,COLORS['PE']),('Temperature-scaled',apply_temperature(p,sel['temperature']),COLORS['threshold'])]:
        edges=np.linspace(0,1,11);centers=[];observed=[];weights=[]
        binid=np.clip(np.searchsorted(edges,q,side='right')-1,0,9)
        for i in range(10):
            mask=binid==i
            if mask.any(): centers.append(float(np.mean(q[mask])));observed.append(float(np.mean(y[mask])));weights.append(int(mask.sum()))
        ax.plot(centers,observed,'o-',lw=1.1,ms=4,label=f'{name}, ECE={ece(y,q):.3f}',color=col)
    ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mean predicted probability',ylabel='Observed positive fraction',
           title='Supplement: reliability diagram (pooled horizons)')
    ax.legend(frameon=False,fontsize=7);ax.grid(alpha=.15)
    fig.tight_layout(); save_fig(fig,'fig_S_reliability')

In [ ]:
# CELL 17 — actual inference benchmarks (no fabricated FLOPs or deployment FPS)
def get_one_example(regime='PE',h=.5):
    dl=get_loader('test',regime,h)
    x,_=next(iter(dl));return x[:1]

def benchmark_one_model(regime,name):
    model=load_model(regime,name)
    x_cpu=get_one_example('PE',.5)
    x=preprocess_uint8_batch(x_cpu,training=False)
    with torch.inference_mode(), amp_context():
        for _ in range(4):
            _=model(x);maybe_sync()
        t_model=[]
        if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
        for _ in range(CFG.LATENCY_REPEATS):
            t=time.perf_counter();_=model(x);maybe_sync();t_model.append((time.perf_counter()-t)*1000)
        peak_mb=float(torch.cuda.max_memory_allocated()/(1024**2)) if DEVICE.type=='cuda' else np.nan
    # Cached uint8 read + normalization + model latency; excludes original video decoding.
    row=CLIPS[(CLIPS.split=='test')&(CLIPS.horizon=='H0.5')].iloc[0]
    t_total=[]
    with torch.inference_mode(), amp_context():
        for _ in range(min(15,CFG.LATENCY_REPEATS)):
            t=time.perf_counter()
            sample=preprocess_uint8_batch(load_uint8_window(row.path, row.window_index)[None],training=False)
            _=model(sample);maybe_sync();t_total.append((time.perf_counter()-t)*1000)
    params=sum(p.numel() for p in model.parameters())
    del model;gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return {'Model':name,'Params_M':params/1e6,'MACs':'not_measured',
            'Model_p50_ms':np.median(t_model),'Model_p95_ms':np.percentile(t_model,95),
            'EndToEnd_p50_ms':np.median(t_total),'EndToEnd_p95_ms':np.percentile(t_total,95),
            'Peak_cuda_allocated_MB':peak_mb,'hardware':str(DEVICE),
            'timing_note':f"Batch size 1, uint8 memmap + preprocessing, AMP={DEVICE.type == 'cuda' and CFG.AMP_ON_CUDA}, warmed up, no camera acquisition"}

def benchmark_pipeline(regime,names):
    # Measures the full MULTI-MODEL forward chain; never sums independently timed latencies.
    models=[load_model(regime,name) for name in names]
    x_cpu=get_one_example('PE',.5)
    x=preprocess_uint8_batch(x_cpu,training=False)
    nums=[sum(p.numel() for p in mdl.parameters()) for mdl in models]
    with torch.inference_mode(), amp_context():
        for _ in range(4):
            for mdl in models: _=mdl(x)
            maybe_sync()
        if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
        elapsed=[]
        for _ in range(CFG.LATENCY_REPEATS):
            t=time.perf_counter()
            for mdl in models: _=mdl(x)
            maybe_sync();elapsed.append((time.perf_counter()-t)*1000)
        peak=(torch.cuda.max_memory_allocated()/(1024**2)) if DEVICE.type=='cuda' else np.nan
        # End-to-end cached-clip reading, normalization, transfer, then all model forwards.
        row=CLIPS[(CLIPS.split=='test')&(CLIPS.horizon=='H0.5')].iloc[0]
        e2e=[]
        for _ in range(min(15,CFG.LATENCY_REPEATS)):
            t=time.perf_counter()
            inp=preprocess_uint8_batch(load_uint8_window(row.path, row.window_index)[None],training=False)
            for mdl in models: _=mdl(inp)
            maybe_sync();e2e.append((time.perf_counter()-t)*1000)
    del models;gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return {'Params_M':sum(nums)/1e6,'MACs':'not_measured',
            'Model_p50_ms':float(np.median(elapsed)), 'Model_p95_ms':float(np.percentile(elapsed,95)),
            'EndToEnd_p50_ms':float(np.median(e2e)), 'EndToEnd_p95_ms':float(np.percentile(e2e,95)),
            'Peak_cuda_allocated_MB':float(peak),'hardware':str(DEVICE),
            'scope':f"uint8 memmap + normalization + sequential branches, batch size 1, AMP={DEVICE.type == 'cuda' and CFG.AMP_ON_CUDA}"}

def benchmark_models():
    if not final_test_gate_is_open(): raise PermissionError('Benchmark test input locked')
    confirm_frozen_selection(SELECTION)
    rows=[benchmark_one_model('PE',m) for m in list(CFG.ENCODERS)+[STATIC_BASELINE]]
    d=pd.DataFrame(rows); d.to_csv(RUN_ROOT/'tables'/'table_S_model_efficiency.csv',index=False)
    selected=[('Best single',SELECTION['best_single']),
              ('Optimized 2-branch',SELECTION['best_two']),
              ('Optimized 3-branch',SELECTION['best_three'])]
    t2=pd.read_csv(RUN_ROOT/'tables'/'table_II_pre_event_models.csv')
    main=[]
    for label,item in selected:
        measures=benchmark_pipeline('PE',item['names'])
        rowname=label if label!='Best single' else None
        if label=='Best single':
            _,y,p=as_horizon_arrays('test',item['names'],item['weights'])
            hm=hmAP(y,p)
            bs=float(np.mean([brier_score_loss(y,p[:,j]) for j in range(3)]))
        else:
            r=t2[t2.Model==label].iloc[0];hm=float(r.h_mAP);bs=float(r.Brier_pooled)
        main.append({'Model':label,'h_mAP':hm,'Brier':bs,**measures})
    main=pd.DataFrame(main)
    main.to_csv(RUN_ROOT/'tables'/'table_III_efficiency_and_fusion.csv',index=False)
    print(main[['Model','h_mAP','Brier','Params_M','Model_p50_ms','EndToEnd_p50_ms']].to_string(index=False))
    return main

def profile_input_pipeline(split='train',regime='PE',batches=20):
    dl=get_loader(split,regime,training=(split=='train'),seed=RUN_SEED)
    tic=time.perf_counter(); count=0; copied_bytes=0
    steps_measured = 0
    for step,(clips,labels) in enumerate(dl):
        if step>=batches:break
        steps_measured += 1
        count+=len(labels); copied_bytes+=clips.numel()*clips.element_size()
        _=preprocess_uint8_batch(clips,training=(split=='train'))
    maybe_sync(); elapsed=max(1e-9,time.perf_counter()-tic)
    row={'seed':RUN_SEED,'batches':steps_measured,'examples':count,
         'clips_per_second_including_first_worker_start':count/elapsed,
         'host_to_device_gib':copied_bytes/(1024**3),'device':str(DEVICE),
         'num_workers':NUM_WORKERS,'pinned_memory':bool(CFG.PIN_MEMORY and DEVICE.type=='cuda'),
         'gpu_augmentation':CFG.GPU_AUGMENT,'cache_backend':CFG.CACHE_BACKEND}
    save_json(row,RUN_ROOT/'logs'/'input_pipeline_profile.json')
    print(row)
    return row


In [ ]:
# CELL 18 — final paper artifacts, optional efficiency benchmarking
if 'report' in CFG.STAGES:
    for training_seed in CFG.RUN_SEEDS:
        activate_seed(training_seed)
        SELECTION=json.loads((RUN_ROOT/'selection'/'selection.lock.json').read_text())
        confirm_frozen_selection(SELECTION)
        if not (RUN_ROOT/'selection'/'final_test_completed.json').exists(): raise RuntimeError('Run locked final test first')
        figure_2_horizons_and_fusion(SELECTION)
        figure_3_threshold_transfer()
        figure_S_calibration(SELECTION)
        # Hardware benchmark is separate to avoid slow repeated inference on every plotting run.
        print('Figures created from ACTUAL independent-test predictions.')
if 'benchmark' in CFG.STAGES:
    for training_seed in CFG.RUN_SEEDS:
        activate_seed(training_seed)
        SELECTION=json.loads((RUN_ROOT/'selection'/'selection.lock.json').read_text())
        benchmark_models()
        tab=pd.read_csv(RUN_ROOT/'tables'/'table_III_efficiency_and_fusion.csv')
        fig,ax=plt.subplots(figsize=(4.2,3.0))
        for _,r in tab.iterrows():
            ax.scatter(r['EndToEnd_p50_ms'],r['h_mAP'],s=47)
            ax.annotate(r['Model'],(r['EndToEnd_p50_ms'],r['h_mAP']),xytext=(3,4),textcoords='offset points',fontsize=7)
        ax.set(xlabel='Measured end-to-end inference latency (ms)',ylabel='Horizon-averaged AP',
               title='Supplement: pre-event discrimination vs runtime')
        ax.grid(alpha=.16);fig.tight_layout();save_fig(fig,'fig_S_accuracy_latency')
if 'profile_cache' in CFG.STAGES:
    for training_seed in CFG.RUN_SEEDS:
        activate_seed(training_seed)
        profile_input_pipeline()


In [ ]:
# CELL 19 — aggregate ALL predefined seeds (no test seed selection, no pseudoreplication)
# Run only AFTER every prespecified seed has been tuned, tested and reported.
def aggregate_all_seeds():
    seeds=list(CFG.TRAIN_SEEDS)
    per=[];tables=[]
    for seed in seeds:
        activate_seed(seed)
        lock_path=RUN_ROOT/'selection'/'selection.lock.json'
        completed=RUN_ROOT/'selection'/'final_test_completed.json'
        if not lock_path.exists() or not completed.exists():
            raise RuntimeError(f'Seed {seed} missing locked selection or final test. Refusing incomplete seed roster')
        sel=json.loads(lock_path.read_text());confirm_frozen_selection(sel)
        tbl_path=RUN_ROOT/'tables'/'table_II_pre_event_models.csv'
        if not tbl_path.exists(): raise FileNotFoundError(f'Run report for seed {seed} first: {tbl_path}')
        d=pd.read_csv(tbl_path);d['seed']=seed;tables.append(d)
        id0,y,A=as_horizon_arrays('test',sel['best_single']['names'],sel['best_single']['weights'])
        id1,y2,B=as_horizon_arrays('test',sel['primary_fusion']['names'],sel['primary_fusion']['weights'])
        if not np.array_equal(id0,id1) or not np.array_equal(y,y2): raise RuntimeError('Misaligned primary test models')
        per.append((id0,y,A,B))
    ids,y=per[0][0],per[0][1]
    for si,(video_ids,labels,_,_) in enumerate(per[1:],1):
        if not np.array_equal(video_ids,ids) or not np.array_equal(labels,y):
            raise RuntimeError('Across-seed test videos/labels differ')
    single=np.stack([p[2] for p in per],axis=0)
    fusion=np.stack([p[3] for p in per],axis=0)
    def seed_average_stat(y,arr):
        return float(np.mean([hmAP(y,p) for p in arr]))
    def paired_seed_bootstrap(X,Y,seed):
        # Resample VIDEOS; keep seed dimension intact. CI is conditional on trained seeds.
        rng=np.random.default_rng(seed)
        observed=seed_average_stat(y,X)-seed_average_stat(y,Y)
        boot=[];extreme=0
        for _ in range(CFG.BOOTSTRAPS):
            idx=stratified_bootstrap_video_indices(y,rng)
            boot.append(seed_average_stat(y[idx],X[:,idx,:])-seed_average_stat(y[idx],Y[:,idx,:]))
        for _ in range(CFG.PERMUTATIONS):
            # Swap paired predictions per video across ALL seeds, preserving seed correlation.
            swap=rng.integers(0,2,size=len(y)).astype(bool)[None,:,None]
            first=np.where(swap,Y,X);second=np.where(swap,X,Y)
            d=seed_average_stat(y,first)-seed_average_stat(y,second)
            extreme+=abs(d)>=abs(observed)-1e-12
        return {'difference_seed_mean':observed,'CI_lower':float(np.quantile(boot,.025)),
                'CI_upper':float(np.quantile(boot,.975)),
                'p_perm_two_sided':float((extreme+1)/(CFG.PERMUTATIONS+1))}
    h1=paired_seed_bootstrap(single[:,:,[0]],single[:,:,[2]],CFG.SEED+801)
    h2=paired_seed_bootstrap(fusion,single,CFG.SEED+802)
    adjust=multipletests([h1['p_perm_two_sided'],h2['p_perm_two_sided']],method='holm')[1]
    for r,name,p in [(h1,'H1',adjust[0]),(h2,'H2',adjust[1])]:
        r['hypothesis']=name;r['p_holm_two_sided']=float(p);r['seeds']=','.join(map(str,seeds))
    out=pd.concat(tables,ignore_index=True)
    metrics=['AP_0.5','AP_1.0','AP_1.5','h_mAP','Brier_pooled','ECE10_pooled']
    summary=out.groupby('Model')[metrics].agg(['mean','std']).reset_index()
    summary.columns=['Model']+[f'{m}_{stat}' for m,stat in summary.columns.tolist()[1:]]
    (ROOT/'tables').mkdir(exist_ok=True)
    out.to_csv(ROOT/'tables'/'multiseed_all_model_results.csv',index=False)
    summary.to_csv(ROOT/'tables'/'multiseed_table_II_mean_sd.csv',index=False)
    tests=pd.DataFrame([h1,h2]);tests.to_csv(ROOT/'stats'/'multiseed_H1_H2.csv',index=False)
    effects=[]
    for i,seed in enumerate(seeds):
        effects.append({'seed':seed,'H1_delta':hmAP(y,single[i][:,[0]])-hmAP(y,single[i][:,[2]]),
                        'H2_delta':hmAP(y,fusion[i])-hmAP(y,single[i]),
                        'best_single_hmAP':hmAP(y,single[i]),'fusion_hmAP':hmAP(y,fusion[i])})
    pd.DataFrame(effects).to_csv(ROOT/'stats'/'multiseed_per_seed_effects.csv',index=False)
    save_json({'seeds':seeds,'split_sha256':sha256_bytes(ROOT/'manifests'/'split_manifest.csv'),
               'inference_scope':'source-video paired, fixed test split, averaged across prespecified seeds',
               'CI_scope':'conditional on observed training seeds; does not quantify independent-split variance',
               'bootstraps':CFG.BOOTSTRAPS,'permutations':CFG.PERMUTATIONS},ROOT/'stats'/'multiseed_method.json')
    # Publication-level Figure 2 is derived from ALL seeds, not cherry-picked runs.
    fig,axs=plt.subplots(1,2,figsize=(7.15,2.8),gridspec_kw={'width_ratios':[1.25,1]})
    subset=['Single-frame','EfficientNet + BiGRU','MobileNetV3 + TSM',
            'Compact3D (X3D-inspired)','Optimized 2-branch','Optimized 3-branch']
    palette=['#708294','#3f72a7','#308d73','#c16d4c','#d18b32','#864c96']
    for lab,col in zip(subset,palette):
        d=out[out.Model==lab].sort_values('seed')
        if d.empty: continue
        vals=d[['AP_0.5','AP_1.0','AP_1.5']].to_numpy(float)
        means=vals.mean(axis=0);std=vals.std(axis=0,ddof=1) if len(vals)>1 else np.zeros(3)
        axs[0].errorbar(CFG.HORIZONS,means,yerr=std,marker='o',lw=1.1,ms=3.2,capsize=2.4,
                        label=lab,color=col,linestyle='--' if lab=='Single-frame' else '-')
    axs[0].set(xlabel='Pre-event horizon (s)',ylabel='Average precision',
               title='(a) Seed-mean discrimination (±SD)')
    axs[0].set_xticks(CFG.HORIZONS);axs[0].set_ylim(0,1.03);axs[0].grid(alpha=.15)
    axs[0].legend(fontsize=6,loc='lower left',frameon=False)
    effects=pd.read_csv(ROOT/'stats'/'multiseed_per_seed_effects.csv')
    d=effects.H2_delta.to_numpy(float)
    axs[1].axvline(0,color='gray',ls='--',lw=1)
    axs[1].scatter(d,np.arange(len(d))+.4,s=22,c='#9a6d37',label='Independent training seeds')
    estimate=h2['difference_seed_mean'];lo=h2['CI_lower'];hi=h2['CI_upper']
    axs[1].errorbar(estimate,len(d)+.25,xerr=[[estimate-lo],[hi-estimate]],fmt='o',
                    color='#864c96',capsize=3,label='Seed-mean Δ (video-bootstrap CI)')
    axs[1].set_yticks(list(np.arange(len(d))+.4)+[len(d)+.25],
                      [str(s) for s in seeds]+['Mean + 95% CI'])
    axs[1].set(xlabel='Fusion − single, horizon-mean AP',title='(b) Prespecified seed effects')
    axs[1].grid(axis='x',alpha=.15)
    fig.tight_layout(pad=1.)
    fig.savefig(ROOT/'figures'/'fig_2_multiseed_horizons.pdf',bbox_inches='tight')
    fig.savefig(ROOT/'figures'/'fig_2_multiseed_horizons.png',dpi=300,bbox_inches='tight')
    plt.close(fig)
    print('MULTI-SEED SUMMARY (mean ± SD):');display(summary)
    print('Confirmatory seed-mean contrasts:');display(tests)
    return summary,tests

if 'aggregate' in CFG.STAGES:
    MULTISEED_SUMMARY,MULTISEED_TESTS=aggregate_all_seeds()


In [ ]:
# CELL 20 — ONE-CLICK orchestrator: all seeds → freeze all tuning → gated test → outputs
# Default: Kaggle "Run All". Existing verified artifacts resume; no silent test re-selection.
from dataclasses import asdict

ORCH_SPEC=ROOT/'manifests'/'one_click_protocol.json'
ORCH_STATUS=ROOT/'one_click_progress.json'


def one_click_protocol():
    # Do not include execution-only switches in the scientific identity.
    excluded={'EXECUTION_MODE','STAGES','RUN_BENCHMARKS','RUN_CACHE_PROFILE',
              'RUN_SEEDS','FINAL_TEST_UNLOCK','ALLOW_OVERWRITE_FINAL',
              'CACHE_RESERVE_GIB','CACHE_MAX_GIB','NUM_WORKERS','VIDEO_WORKERS',
              'PERSISTENT_WORKERS','PREFETCH_FACTOR','PIN_MEMORY',
              'LATENCY_REPEATS'}
    values={k:v for k,v in asdict(CFG).items() if k not in excluded}
    values['orchestration_version']='ONE_CLICK_V1'
    return values


def ensure_one_click_protocol():
    spec=one_click_protocol()
    if ORCH_SPEC.exists():
        existing=json.loads(ORCH_SPEC.read_text())
        if existing != json.loads(json.dumps(spec)):
            raise RuntimeError('One-click scientific configuration differs from the existing WORK_DIR. '
                               'Restore the original configuration or use a NEW WORK_DIR.')
    else:
        # Do not import an unverified previous experiment as a fresh one-click run.
        if (ROOT/'manifests'/'split_manifest.csv').exists():
            raise RuntimeError('Existing split manifest has no one-click protocol record. '
                               'Use a fresh WORK_DIR; do not silently adopt older research artifacts.')
        save_json(spec,ORCH_SPEC)
    return spec


def ensure_audit_once():
    global MANIFEST
    path=ROOT/'manifests'/'split_manifest.csv'
    summ=ROOT/'manifests'/'audit_summary.json'
    if not path.exists():
        MANIFEST=build_audit()
        print('AUDIT: new frozen source-video split created')
    else:
        if not summ.exists():
            raise RuntimeError('Partial audit: split exists without audit summary. Investigate the interrupted audit.')
        info=json.loads(summ.read_text())
        if info.get('protocol')!='NEXAR_TVDR_V1' or info.get('seed')!=CFG.SEED:
            raise RuntimeError('Saved audit protocol/seed differs; refusing silent split reuse')
        MANIFEST=pd.read_csv(path,dtype={'video_id':str})
        print('AUDIT: reusing previously frozen split')
    if (MANIFEST.empty or MANIFEST.video_id.isna().any() or
        not MANIFEST.video_id.is_unique or
        set(MANIFEST.split.unique())!={'train','tune','test'} or
        set(MANIFEST.target.unique())!={0,1} or
        not MANIFEST.apply(lambda r:valid_anchor(r.anchor_time,r.duration),axis=1).all()):
        raise RuntimeError('Split manifest failed reuse validation')
    if summ.exists() and int(json.loads(summ.read_text()).get('eligible',-1))!=len(MANIFEST):
        raise RuntimeError('Audit summary and split manifest row counts disagree')
    return MANIFEST


def ensure_cache_once():
    global CLIPS
    path=ROOT/'manifests'/'clip_manifest.csv'
    if not path.exists():
        CLIPS=build_clips()  # individual cached videos already resume safely
        print('EXTRACT: cache and clip manifest completed')
    else:
        CLIPS=pd.read_csv(path,dtype={'video_id':str})
        spec=ROOT/'manifests'/'cache_spec.json'
        if not spec.is_file() or json.loads(spec.read_text()).get('cache_schema')!=CACHE_SCHEMA:
            raise RuntimeError('Cache protocol mismatch; refusing stale cache')
        if ('cache_dtype' not in CLIPS or set(CLIPS.cache_dtype)!={'uint8'} or
            CLIPS.video_id.nunique()!=MANIFEST.video_id.nunique() or
            len(CLIPS)!=len(MANIFEST)*len(WINDOWS) or
            CLIPS.duplicated(['video_id','horizon']).any() or
            set(CLIPS.video_id)!=set(MANIFEST.video_id) or
            set(CLIPS.horizon)!=set(map(clip_key,WINDOWS))):
            raise RuntimeError('Clip manifest is missing, duplicated or incompatible')
        manifest_by_id={str(r.video_id):r for r in MANIFEST.itertuples(index=False)}
        for row in CLIPS.itertuples(index=False):
            r=manifest_by_id[str(row.video_id)]
            if row.split!=r.split or int(row.target)!=int(r.target):
                raise RuntimeError('Clip manifest contradicts source-video split')
        bad=[vid for vid,r in manifest_by_id.items() if not valid_cached_video(r)]
        if bad: raise RuntimeError(f'{len(bad)} stale/unverifiable caches (first: {bad[:3]}). '
                                   'Correct cache source or rerun extraction before proceeding.')
        print('EXTRACT: verified and reused existing uint8 cache')
    return CLIPS


def one_click_inference_specs():
    specs=[('EC',m,h) for m in CFG.ENCODERS for h in ('EC',)+CFG.HORIZONS]
    specs += [('PE',m,h) for m in list(CFG.ENCODERS)+[STATIC_BASELINE] for h in CFG.HORIZONS]
    return specs


def assert_final_test_is_sealed():
    # A completion record must match the *current frozen lock* and every prediction.
    complete=RUN_ROOT/'selection'/'final_test_completed.json'
    lock=RUN_ROOT/'selection'/'selection.lock.json'
    if not complete.exists():
        raise RuntimeError('No completed final inference record')
    doc=json.loads(complete.read_text())
    if (doc.get('selection_sha256')!=sha256_bytes(lock) or
        doc.get('pilot_only')!=CFG.DRY_RUN):
        raise RuntimeError('Final-test completion record no longer matches the frozen selection')
    manifest_digest=sha256_bytes(ROOT/'manifests'/'clip_manifest.csv')
    checkpoint_digests={}
    for regime,name,h in one_click_inference_specs():
        path=inference_path('test',regime,name,h)
        meta_path=path.with_suffix('.meta.json')
        if not path.is_file() or not meta_path.is_file():
            raise RuntimeError(f'Final-test completion marker exists but prediction is missing: {path}')
        key=(regime,name)
        if key not in checkpoint_digests:
            checkpoint_digests[key]=sha256_bytes(checkpoint_path(regime,name))
        expected={'checkpoint_sha256':checkpoint_digests[key],
                  'clip_manifest_sha256':manifest_digest,
                  'split':'test','regime':regime,'name':name,'horizon':clip_key(h),
                  'prediction_sha256':sha256_bytes(path)}
        saved=json.loads(meta_path.read_text())
        if any(saved.get(k)!=v for k,v in expected.items()):
            raise RuntimeError(f'Final-test prediction has invalid provenance: {path}')
    return True


def persist_one_click_progress(phase, **details):
    save_json({'phase':phase,'updated_utc':datetime.now(timezone.utc).isoformat(),
               'pilot_only':CFG.DRY_RUN,'seeds':list(CFG.TRAIN_SEEDS),**details},ORCH_STATUS)
    print('ONE-CLICK:',phase)


def run_one_click():
    global MANIFEST, CLIPS, SELECTION, FINAL_TABLE, CONFIRMATORY, MULTISEED_SUMMARY, MULTISEED_TESTS
    if CFG.EXECUTION_MODE!='one_click':
        raise RuntimeError('This function is for EXECUTION_MODE=one_click only')
    ensure_one_click_protocol()
    # A confirmatory test must never coexist with an unfrozen seed selection.
    frozen_missing=[seed for seed in CFG.TRAIN_SEEDS if not
                    (ROOT/'runs'/f'seed_{seed}'/'selection'/'selection.lock.json').exists()]
    if frozen_missing and any(ROOT.glob('runs/seed_*/predictions/test__*.csv')):
        raise RuntimeError('Test predictions exist while at least one prespecified seed lacks a frozen tuning lock. '
                           'Cannot safely resume a confirmatory experiment.')
    persist_one_click_progress('audit')
    ensure_audit_once()
    protocol_self_checks()
    persist_one_click_progress('cache')
    ensure_cache_once()
    # ALL model training precedes ANY final-test inference; predeclared seed roster unchanged.
    training_jobs=[('EC',m) for m in CFG.ENCODERS]
    training_jobs += [('PE',m) for m in CFG.ENCODERS]+[('PE',STATIC_BASELINE)]
    for seed in CFG.TRAIN_SEEDS:
        activate_seed(seed)
        persist_one_click_progress('training',seed=seed)
        for regime,name in training_jobs:
            train_model(regime,name)  # completes once or resumes saved epoch states
    # Never regenerate tuning predictions after a lock already exists.
    for seed in CFG.TRAIN_SEEDS:
        activate_seed(seed)
        persist_one_click_progress('tuning_and_lock',seed=seed)
        lock=RUN_ROOT/'selection'/'selection.lock.json'
        if lock.exists():
            SELECTION=json.loads(lock.read_text())
            confirm_frozen_selection(SELECTION)
            print('TUNING: existing selection is immutable for seed',seed)
        else:
            if any((RUN_ROOT/'predictions').glob('test__*.csv')):
                raise RuntimeError('Test predictions exist before tuning was locked. '
                                   'Cannot proceed with a valid confirmatory selection.')
            infer_split('tune')
            SELECTION=create_tuning_lock()
            confirm_frozen_selection(SELECTION)
    # Gate remains CLOSED until all seeds are frozen AND all content hashes agree.
    persist_one_click_progress('verify_all_frozen_selections')
    seal_and_open_one_click_gate()
    # Resumable test: partial prediction files may be verified/reused; completed
    # test sets are never recomputed or silently overwritten.
    for seed in CFG.TRAIN_SEEDS:
        activate_seed(seed)
        SELECTION=json.loads((RUN_ROOT/'selection'/'selection.lock.json').read_text())
        confirm_frozen_selection(SELECTION)
        if not final_test_gate_is_open():
            raise PermissionError('Final test gate unexpectedly closed')
        persist_one_click_progress('test_and_reporting',seed=seed)
        completed=RUN_ROOT/'selection'/'final_test_completed.json'
        if completed.exists():
            assert_final_test_is_sealed()
            print('TEST: immutable completed predictions verified for seed',seed)
        else:
            infer_split('test')  # cached partial predictions verified on reuse
            save_json({'final_inference_completed_utc':datetime.now(timezone.utc).isoformat(),
                       'selection_sha256':sha256_bytes(RUN_ROOT/'selection'/'selection.lock.json'),
                       'pilot_only':CFG.DRY_RUN},completed)
            assert_final_test_is_sealed()
            print('TEST: completed and sealed for seed',seed)
        FINAL_TABLE,CONFIRMATORY=main_tables(SELECTION)
        figure_2_horizons_and_fusion(SELECTION)
        figure_3_threshold_transfer()
        figure_S_calibration(SELECTION)
        if CFG.RUN_BENCHMARKS:
            benchmark_models()
            tab=pd.read_csv(RUN_ROOT/'tables'/'table_III_efficiency_and_fusion.csv')
            fig,ax=plt.subplots(figsize=(4.2,3.0))
            for _,r in tab.iterrows():
                ax.scatter(r['EndToEnd_p50_ms'],r['h_mAP'],s=47)
                ax.annotate(r['Model'],(r['EndToEnd_p50_ms'],r['h_mAP']),
                            xytext=(3,4),textcoords='offset points',fontsize=7)
            ax.set(xlabel='Measured end-to-end inference latency (ms)',
                   ylabel='Horizon-averaged AP',title='Supplement: pre-event discrimination vs runtime')
            ax.grid(alpha=.16)
            fig.tight_layout()
            save_fig(fig,'fig_S_accuracy_latency')
        if CFG.RUN_CACHE_PROFILE:
            profile_input_pipeline()
    persist_one_click_progress('across_seed_aggregation')
    MULTISEED_SUMMARY,MULTISEED_TESTS=aggregate_all_seeds()
    # Scope of outputs is explicit; this does NOT claim real-world deployment validation.
    inventory={k:len(list(ROOT.glob(pattern))) for k,pattern in {
        'tables':'**/tables/*.csv','stats':'**/stats/*.csv',
        'figures_pdf':'**/figures/*.pdf','figures_png':'**/figures/*.png'}.items()}
    persist_one_click_progress('complete',generated_artifact_counts=inventory)
    print('COMPLETE: research outputs under',ROOT)
    return MULTISEED_SUMMARY,MULTISEED_TESTS


if CFG.EXECUTION_MODE == 'one_click':
    MULTISEED_SUMMARY, MULTISEED_TESTS = run_one_click()
